# Proximity-Preserving Neural SubdivisionReference implementation

> H. Ugail, *Proximity-Preserving Neural Subdivision*, 2026.

This single notebook is the complete pipeline. It builds the constrainedsubdivision operator, certifies the architectural guarantees numerically,ablates each architectural element, trains the operator on the Gaussian-ridgebenchmark, and regenerates every figure and metric table reported in the paper.

The operator refines a triangle mesh by applying Loop's one-to-four split,keeping Loop's old-vertex update, and inserting a learned edge vertex$$q_i^{\theta} \;=\; q_i^{0} \;+\; h_i^{2}\,\gamma_i\,F_i\,\eta_{\theta}(\varphi_i),$$where $q_i^{0}$ is the Loop edge vertex, $h_i$ the local edge length, $\gamma_i\in [0,1]$ a curvature gate that vanishes to second order on planar input,$F_i \in SO(3)$ a covariant local frame, and $\eta_{\theta}$ a small multilayerperceptron whose output is hard-bounded by $\|\eta_\theta\| \le C$.

Every structural property follows from this parameterisation and holds for any finitenetwork weights, before any training takes place.

**Running the notebook.** Run all cells. Checkpoints found in `Results/models`are loaded and their training is skipped, so the default run on a fresh clone isevaluation-only and needs no GPU. Deleting a checkpoint retrains exactly thatmodel, and emptying `Results/models` retrains the whole system. For a quickfunctional check set `SMOKE_MODE = True` in the configuration cell below.

## 0. ConfigurationEvery path is derived from the repository root, so nothing needs editing beforethe first run. Figures are written to `Results/figures`, metric tables to`Results/metrics`, and checkpoints to `Results/models`.

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
# Every path below is derived from the repository root, so this
# notebook runs unchanged after a plain `git clone`, both locally
# and in Google Colab. There is no external storage to mount and
# no directory to edit before running.

import os
from pathlib import Path


def _find_repo_root(start=None):
    """Locate the repository root by walking upwards from the working directory."""
    here = Path(start or os.getcwd()).resolve()
    for candidate in [here] + list(here.parents):
        if (candidate / 'run_pipeline_full.ipynb').exists() or (candidate / '.git').is_dir():
            return candidate
    return here


# ---- Run controls ------------------------------------------------------
# SMOKE_MODE    : tiny datasets and short training, for a functional check.
#                 A smoke run writes to Results/_smoke and never touches the
#                 shipped checkpoints, figures, or tables.
# FORCE_RETRAIN : ignore the shipped checkpoints and retrain everything.
SMOKE_MODE    = False
FORCE_RETRAIN = False

REPO_ROOT   = _find_repo_root()
RESULTS_DIR = REPO_ROOT / 'Results'
MODEL_DIR   = RESULTS_DIR / 'models'
_out_root   = (RESULTS_DIR / '_smoke') if SMOKE_MODE else RESULTS_DIR
FIG_DIR     = _out_root / 'figures'
METRIC_DIR  = _out_root / 'metrics'
for _d in (FIG_DIR, METRIC_DIR, MODEL_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"Repository root : {REPO_ROOT}")
print(f"Figures         -> {FIG_DIR.relative_to(REPO_ROOT)}")
print(f"Metric tables   -> {METRIC_DIR.relative_to(REPO_ROOT)}")
print(f"Checkpoints     -> {MODEL_DIR.relative_to(REPO_ROOT)}")
print(f"smoke_mode={SMOKE_MODE}   force_retrain={FORCE_RETRAIN}")

### Imports and global settingsThe gate sensitivity `C_GATE` is a hyperparameter of the feature map and entersnone of the structural results.

In [ ]:
import math
import json
import time
import csv
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

DTYPE = torch.float32
torch.set_default_dtype(DTYPE)
SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)

# `OUT` is the figure directory. Metric tables and checkpoints are written to
# METRIC_DIR and MODEL_DIR respectively, all inside Results/.
OUT = FIG_DIR

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11.5, "axes.titleweight": "normal",
    "axes.labelsize": 10.5,
    "xtick.labelsize": 9.5, "ytick.labelsize": 9.5,
    "legend.fontsize": 9, "legend.framealpha": 0.95,
    "legend.edgecolor": "0.85", "legend.facecolor": "white",
    "figure.dpi": 110, "savefig.dpi": 180,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "lines.linewidth": 1.6, "lines.markersize": 5,
    "axes.axisbelow": True,
})

PALETTE = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

# Gate sensitivity c in gamma_i = tanh(c ||phi_curv||^2). This is a
# hyperparameter of the feature map and affects no theorem.
C_GATE = 10.0

# Collects every number the notebook produces; written to Results/metrics at the end.
RESULTS = defaultdict(dict)


def to_safe(o):
    """Recursively convert numpy and torch objects into JSON-serialisable ones."""
    if isinstance(o, dict):  return {str(k): to_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):  return [to_safe(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, torch.Tensor): return o.detach().cpu().numpy().tolist()
    return o


def write_table(name, fieldnames, rows):
    """Write a metric table to Results/metrics/<name>.csv and echo the path."""
    path = METRIC_DIR / f'{name}.csv'
    with open(path, 'w', newline='') as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            writer.writerow({k: r.get(k, '') for k in fieldnames})
    print(f"  table -> Results/metrics/{name}.csv")
    return path


print(f"torch={torch.__version__}  numpy={np.__version__}  dtype={DTYPE}  c_gate={C_GATE}")

## 1. The PNS operatorThe operator is written as an additive perturbation of Loop, $S_\theta = S_0 +R_\theta$. Four ingredients carry the structural roles. The intrinsic featurevector $\varphi_i$ collects the curvature descriptor $1 - n_{f_1} \cdot n_{f_2}$together with four logarithmic edge-length ratios, all invariant under rigidmotion. The gate $\gamma_i = \tanh(c\,\|\varphi_i^{\mathrm{curv}}\|^2)$ has a zeroof order two at planar configurations. The frame $F_i = [T_i, N_i, B_i]$transforms covariantly. The network output is projected onto the ball of radius$C$, so the bound holds by construction rather than by penalty.The switches `use_gate`, `use_frame` and `use_h2` remove one architecturalelement at a time and are used by the ablation in Section 8. The subdivisionroutines at the end of the cell apply the operator to its own output, withboundary edges always routed through Loop's boundary midpoint so the refinedtopology matches `subdivide_loop` exactly.

In [ ]:
class Mesh:
    def __init__(self, V, F):
        self.V = V if isinstance(V, torch.Tensor) else torch.tensor(V, dtype=DTYPE)
        self.F = F if isinstance(F, torch.Tensor) else torch.tensor(F, dtype=torch.long)
        if self.V.dtype != DTYPE: self.V = self.V.to(DTYPE)
        if self.F.dtype != torch.long: self.F = self.F.to(torch.long)
    @property
    def n_v(self): return int(self.V.shape[0])
    @property
    def n_f(self): return int(self.F.shape[0])
    def clone(self): return Mesh(self.V.clone(), self.F.clone())


def build_edge_data(F):
    bag = defaultdict(list)
    for fi, (a, b, c) in enumerate(F.tolist()):
        for u, v, opp in ((a, b, c), (b, c, a), (c, a, b)):
            key = (u, v) if u < v else (v, u)
            bag[key].append((fi, opp))
    edges, faces, opps = [], [], []
    for k, lst in bag.items():
        if len(lst) == 2:
            edges.append(list(k))
            faces.append([lst[0][0], lst[1][0]])
            opps.append([lst[0][1], lst[1][1]])
    return (torch.tensor(edges, dtype=torch.long),
            torch.tensor(faces, dtype=torch.long),
            torch.tensor(opps,  dtype=torch.long))


def loop_edge_vertices(V, edges, opps):
    pa, pb = V[edges[:, 0]], V[edges[:, 1]]
    pc, pd = V[opps[:, 0]],  V[opps[:, 1]]
    return 3/8 * (pa + pb) + 1/8 * (pc + pd)


def _face_normals(V, F, eps=1e-10):
    v0, v1, v2 = V[F[:, 0]], V[F[:, 1]], V[F[:, 2]]
    n = torch.cross(v1 - v0, v2 - v0, dim=1)
    return n / torch.norm(n, dim=1, keepdim=True).clamp_min(eps)


def edge_frames(V, F, edges, edge_faces, eps=1e-10):
    pa = V[edges[:, 0]]; pb = V[edges[:, 1]]
    e_vec = pb - pa
    e_len = torch.norm(e_vec, dim=1).clamp_min(eps)
    T = e_vec / e_len.unsqueeze(1)
    Fn = _face_normals(V, F, eps=eps)
    n_avg = Fn[edge_faces[:, 0]] + Fn[edge_faces[:, 1]]
    n_avg = n_avg / torch.norm(n_avg, dim=1, keepdim=True).clamp_min(eps)
    N = n_avg - (n_avg * T).sum(dim=1, keepdim=True) * T
    N = N / torch.norm(N, dim=1, keepdim=True).clamp_min(eps)
    B = torch.cross(T, N, dim=1)
    return torch.stack([T, N, B], dim=2), e_len


def curvature_features(V, F, edges, edge_faces, eps=1e-10):
    Fn = _face_normals(V, F, eps=eps)
    return (1.0 - (Fn[edge_faces[:, 0]] * Fn[edge_faces[:, 1]]).sum(dim=1)).unsqueeze(1)


def shape_features(V, edges, opps, e_len, eps=1e-10):
    pa, pb = V[edges[:, 0]], V[edges[:, 1]]
    pc, pd = V[opps[:, 0]], V[opps[:, 1]]
    e = e_len.clamp_min(eps)
    return torch.stack([
        torch.log(torch.norm(pa - pc, dim=1).clamp_min(eps) / e),
        torch.log(torch.norm(pb - pc, dim=1).clamp_min(eps) / e),
        torch.log(torch.norm(pa - pd, dim=1).clamp_min(eps) / e),
        torch.log(torch.norm(pb - pd, dim=1).clamp_min(eps) / e),
    ], dim=1)


def gate(phi_curv, c=C_GATE):
    return torch.tanh(c * (phi_curv ** 2).sum(dim=1))


class CorrectionNet(nn.Module):
    def __init__(self, dim_in=5, hidden=32, n_layers=2, C=0.5):
        super().__init__()
        layers = []
        d = dim_in
        for _ in range(n_layers):
            layers += [nn.Linear(d, hidden), nn.GELU()]
            d = hidden
        layers += [nn.Linear(hidden, 3)]
        self.net = nn.Sequential(*layers)
        self.C = C
        self.bounded = True
    def forward(self, phi):
        z = self.net(phi)
        if self.bounded:
            n = torch.norm(z, dim=1, keepdim=True)
            return self.C * z / torch.clamp(n, min=1.0)
        return z


def constrained_edge_vertices(mesh, net, c_gate=C_GATE, return_aux=False,
                              use_gate=True, use_frame=True, use_h2=True):
    edges, edge_faces, opps = build_edge_data(mesh.F)
    q0 = loop_edge_vertices(mesh.V, edges, opps)
    frames, e_len = edge_frames(mesh.V, mesh.F, edges, edge_faces)
    phi_curv = curvature_features(mesh.V, mesh.F, edges, edge_faces)
    phi_shape = shape_features(mesh.V, edges, opps, e_len)
    phi = torch.cat([phi_curv, phi_shape], dim=1)
    g = gate(phi_curv, c=c_gate).unsqueeze(1) if use_gate else torch.ones(phi.shape[0], 1)
    eta = net(phi)
    corr_local = (g * eta).unsqueeze(2)
    corr_world = (frames @ corr_local).squeeze(2) if use_frame else corr_local.squeeze(2)
    scale = (e_len ** 2 if use_h2 else e_len).unsqueeze(1)
    q_theta = q0 + scale * corr_world
    if return_aux:
        return q_theta, q0, dict(edges=edges, edge_faces=edge_faces, opposites=opps,
                                  frames=frames, e_len=e_len, phi=phi, gate=g.squeeze(1),
                                  eta=eta, q0=q0)
    return q_theta, q0


def discrete_normal_at_inserts(V, F, q_inserts, edges, edge_faces, eps=1e-9):
    Fn = _face_normals(V, F, eps=eps)
    n_left  = Fn[edge_faces[:, 0]]
    n_right = Fn[edge_faces[:, 1]]
    pa = V[edges[:, 0]]; pb = V[edges[:, 1]]
    a_idx = edges[:, 0]; b_idx = edges[:, 1]
    F_left  = F[edge_faces[:, 0]]
    F_right = F[edge_faces[:, 1]]
    not_ab_left  = ~((F_left  == a_idx.unsqueeze(1)) | (F_left  == b_idx.unsqueeze(1)))
    not_ab_right = ~((F_right == a_idx.unsqueeze(1)) | (F_right == b_idx.unsqueeze(1)))
    c_idx = F_left[not_ab_left]
    d_idx = F_right[not_ab_right]
    pc = V[c_idx]; pd = V[d_idx]
    n1 = torch.cross(q_inserts - pa, pc - pa, dim=1)
    n2 = torch.cross(pb - q_inserts, pc - q_inserts, dim=1)
    n3 = torch.cross(q_inserts - pb, pd - pb, dim=1)
    n4 = torch.cross(pa - q_inserts, pd - q_inserts, dim=1)
    def _sign_match(n, ref):
        return torch.where((n * ref).sum(dim=1, keepdim=True) >= 0, n, -n)
    n1 = _sign_match(n1, n_left); n2 = _sign_match(n2, n_left)
    n3 = _sign_match(n3, n_right); n4 = _sign_match(n4, n_right)
    n_total = n1 + n2 + n3 + n4
    return n_total / n_total.norm(dim=1, keepdim=True).clamp_min(eps)


# Loop subdivision (for repeated-application tests later)
def loop_old_vertex_update(V, F):
    n_v = V.shape[0]
    edge_keys, _, edge_opps = _build_full_edge_index(F)
    boundary_neighbors = {v: set() for v in range(n_v)}
    for k, opps in zip(edge_keys, edge_opps):
        if len(opps) == 1:
            a, b = k
            boundary_neighbors[a].add(b); boundary_neighbors[b].add(a)
    is_bd = [len(boundary_neighbors[v]) > 0 for v in range(n_v)]
    one_ring = [set() for _ in range(n_v)]
    for f in F.tolist():
        a, b, c = f
        one_ring[a].update([b, c]); one_ring[b].update([a, c]); one_ring[c].update([a, b])
    new_V = V.clone()
    for v in range(n_v):
        if is_bd[v]:
            bn = list(boundary_neighbors[v])
            if len(bn) == 2:
                new_V[v] = 0.75 * V[v] + 0.125 * (V[bn[0]] + V[bn[1]])
            continue
        ring = list(one_ring[v]); k = len(ring)
        if k == 0: continue
        beta = 3.0/16.0 if k == 3 else (1.0/k) * (5.0/8.0 - (3.0/8.0 + 0.25*math.cos(2*math.pi/k))**2)
        new_V[v] = (1 - k*beta) * V[v] + beta * V[ring].sum(dim=0)
    return new_V


def _build_full_edge_index(F):
    edge_to_id, edge_keys, edge_opps = {}, [], []
    for (a, b, c) in F.tolist():
        for u, v, opp in ((a, b, c), (b, c, a), (c, a, b)):
            key = (u, v) if u < v else (v, u)
            if key not in edge_to_id:
                edge_to_id[key] = len(edge_keys); edge_keys.append(key); edge_opps.append([opp])
            else:
                edge_opps[edge_to_id[key]].append(opp)
    return edge_keys, edge_to_id, edge_opps


def _new_edge_vertices_loop(V, edge_keys, edge_opps):
    out = torch.zeros(len(edge_keys), 3, dtype=V.dtype)
    for ei, (a, b) in enumerate(edge_keys):
        opps = edge_opps[ei]
        if len(opps) == 2:
            c, d = opps
            out[ei] = 3/8 * (V[a] + V[b]) + 1/8 * (V[c] + V[d])
        else:
            out[ei] = 0.5 * (V[a] + V[b])
    return out


def _topological_subdivide(mesh, new_edge_v):
    """Construct the refined topology with given new edge vertex positions."""
    V, F = mesh.V, mesh.F; n_v = V.shape[0]
    edge_keys, edge_to_id, _ = _build_full_edge_index(F)
    new_old_v = loop_old_vertex_update(V, F)
    new_F = []
    for f in F.tolist():
        a, b, c = f
        ab = n_v + edge_to_id[(a, b) if a<b else (b, a)]
        bc = n_v + edge_to_id[(b, c) if b<c else (c, b)]
        ca = n_v + edge_to_id[(c, a) if c<a else (a, c)]
        new_F.extend([[a,ab,ca], [b,bc,ab], [c,ca,bc], [ab,bc,ca]])
    return Mesh(torch.cat([new_old_v, new_edge_v], dim=0), torch.tensor(new_F, dtype=torch.long))


def subdivide_loop(mesh):
    edge_keys, _, edge_opps = _build_full_edge_index(mesh.F)
    new_edge_v = _new_edge_vertices_loop(mesh.V, edge_keys, edge_opps)
    return _topological_subdivide(mesh, new_edge_v)


def subdivide_constrained(mesh, net, c_gate=C_GATE):
    """Apply the trained constrained operator on interior edges; boundary edges
    get the classical Loop boundary midpoint. This keeps the topology consistent
    with subdivide_loop and keeps the boundary unmodified by the network."""
    edge_keys, edge_to_id, edge_opps = _build_full_edge_index(mesh.F)
    new_edge_v = _new_edge_vertices_loop(mesh.V, edge_keys, edge_opps)
    edges_int, _, _ = build_edge_data(mesh.F)
    with torch.no_grad():
        q_theta, _ = constrained_edge_vertices(mesh, net, c_gate=c_gate)
    for ei in range(edges_int.shape[0]):
        a, b = int(edges_int[ei, 0]), int(edges_int[ei, 1])
        key = (a, b) if a < b else (b, a)
        new_edge_v[edge_to_id[key]] = q_theta[ei]
    return _topological_subdivide(mesh, new_edge_v)


def subdivide_unconstrained(mesh, net):
    """Same boundary treatment as subdivide_constrained: only interior edges
    get the learned correction; boundaries use Loop boundary midpoints."""
    edge_keys, edge_to_id, edge_opps = _build_full_edge_index(mesh.F)
    new_edge_v = _new_edge_vertices_loop(mesh.V, edge_keys, edge_opps)
    edges_int, _, _ = build_edge_data(mesh.F)
    with torch.no_grad():
        q_t, _ = net(mesh)
    for ei in range(edges_int.shape[0]):
        a, b = int(edges_int[ei, 0]), int(edges_int[ei, 1])
        key = (a, b) if a < b else (b, a)
        new_edge_v[edge_to_id[key]] = q_t[ei]
    return _topological_subdivide(mesh, new_edge_v)

## 2. Loss components and training utilitiesThe training loss combines a signed-distance term, a normal-alignment term, afairness term penalising high-frequency normal variation, and a soft proximityregulariser. The proximity term is not the source of the proximity bound, whichis architectural. It only discourages the network from spending the fullenvelope where a smaller correction suffices.`UnconstrainedNet` is the foil. It predicts an unbounded vertex offset directlyfrom raw one-ring coordinates, with no gate, no frame, and no envelope. It is acontrol that isolates the effect of removing the architectural constraints, nota production method.

In [ ]:
def loss_sdf(q_t, sdf_fn):
    return (sdf_fn(q_t) ** 2).mean()


def surface_normal(sdf_fn, p, eps=1e-6):
    p_g = p.detach().clone().requires_grad_(True)
    s = sdf_fn(p_g).sum()
    g, = torch.autograd.grad(s, p_g, create_graph=False)
    return g / g.norm(dim=-1, keepdim=True).clamp_min(eps)


def loss_normal(q_t, n_theta, sdf_fn):
    n_target = surface_normal(sdf_fn, q_t).detach()
    return (1.0 - (n_theta * n_target).sum(dim=1)).mean()


def loss_fairness(n_theta):
    diffs = n_theta[1:] - n_theta[:-1]
    return (diffs ** 2).sum(dim=1).mean()


def loss_proximity_usage(q_t, q_0, e_len, C):
    h2 = (e_len ** 2).clamp_min(1e-12)
    ratio = (q_t - q_0).norm(dim=1) / (C * h2)
    return (ratio ** 2).mean()


def total_loss(mesh, net, sdf_fn, C, weights, c_gate=C_GATE,
               use_gate=True, use_frame=True, use_h2=True):
    q_t, q_0, aux = constrained_edge_vertices(
        mesh, net, c_gate=c_gate, return_aux=True,
        use_gate=use_gate, use_frame=use_frame, use_h2=use_h2)
    n_t = discrete_normal_at_inserts(mesh.V, mesh.F, q_t, aux['edges'], aux['edge_faces'])
    L_sdf = loss_sdf(q_t, sdf_fn)
    L_n   = loss_normal(q_t, n_t, sdf_fn)
    L_f   = loss_fairness(n_t)
    L_p   = loss_proximity_usage(q_t, q_0, aux['e_len'], C)
    L = weights['sdf']*L_sdf + weights['n']*L_n + weights['f']*L_f + weights['p']*L_p
    return L, dict(sdf=float(L_sdf), normal=float(L_n), fair=float(L_f), prox=float(L_p), total=float(L))


# Tuned weights for feature-sensitive task
RIDGE_WEIGHTS = dict(sdf=5.0, n=0.5, f=1e-3, p=1e-4)


def train_operator(train_samples, val_samples, C=0.5, c_gate=C_GATE,
                   weights=None, n_steps=200, lr=2e-3, hidden=32, n_layers=2,
                   bounded=True, use_gate=True, use_frame=True, use_h2=True,
                   verbose=True, seed=0, log_every=50):
    if weights is None: weights = RIDGE_WEIGHTS
    torch.manual_seed(seed)
    net = CorrectionNet(dim_in=5, hidden=hidden, n_layers=n_layers, C=C)
    net.bounded = bounded
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    history = []
    for step in range(n_steps):
        opt.zero_grad()
        breakdown_sums = defaultdict(float)
        for s in train_samples:
            L, parts = total_loss(s['mesh'], net, s['sdf'], C, weights, c_gate=c_gate,
                                   use_gate=use_gate, use_frame=use_frame, use_h2=use_h2)
            L.backward()
            for k, v in parts.items(): breakdown_sums[k] += v
        opt.step()
        avg = {k: v/len(train_samples) for k, v in breakdown_sums.items()}
        history.append(avg)
        if verbose and (step % log_every == 0 or step == n_steps-1):
            print(f"  step {step:4d}: total={avg['total']:.5f}  sdf={avg['sdf']:.5f}  "
                  f"normal={avg['normal']:.4f}  prox={avg['prox']:.5f}")
    return net, history


# Unconstrained baseline (foil)
class UnconstrainedNet(nn.Module):
    def __init__(self, hidden=32, n_layers=3):
        super().__init__()
        layers = [nn.Linear(12, hidden), nn.GELU()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden, hidden), nn.GELU()]
        layers += [nn.Linear(hidden, 3)]
        self.net = nn.Sequential(*layers)
    def forward(self, mesh):
        edges, _, opps = build_edge_data(mesh.F)
        pa = mesh.V[edges[:, 0]]; pb = mesh.V[edges[:, 1]]
        pc = mesh.V[opps[:, 0]];  pd = mesh.V[opps[:, 1]]
        feat = torch.cat([pa, pb, pc, pd], dim=1)
        delta = self.net(feat)
        q0 = 3/8 * (pa + pb) + 1/8 * (pc + pd)
        return q0 + delta, q0


def train_unconstrained(train_samples, n_steps=200, lr=2e-3, seed=0):
    torch.manual_seed(seed)
    net = UnconstrainedNet(hidden=32, n_layers=3)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    history = []
    for step in range(n_steps):
        opt.zero_grad()
        total = 0.0
        for s in train_samples:
            q_t, _ = net(s['mesh'])
            L = (s['sdf'](q_t) ** 2).mean()
            L.backward()
            total += float(L)
        opt.step()
        history.append(total / len(train_samples))
    return net, history

### Checkpoint handling`cached_model` makes the notebook incremental. A model is trained once, saved to`Results/models`, and reused on every later run.

In [ ]:
def steps(n):
    """Training-step count, shortened when SMOKE_MODE is on."""
    return max(5, n // 25) if SMOKE_MODE else n


def cached_model(name, build_fn, train_fn, meta=None):
    """Return a trained network, reusing Results/models/<name>.pt when it exists.

    Deleting a single checkpoint file retrains exactly that model, and emptying
    Results/models retrains the whole system. Setting FORCE_RETRAIN = True in the
    configuration cell has the same effect without deleting anything.
    """
    path = MODEL_DIR / f'{name}.pt'
    if path.exists() and not FORCE_RETRAIN and not SMOKE_MODE:
        net = build_fn()
        ckpt = torch.load(path, map_location='cpu', weights_only=False)
        net.load_state_dict(ckpt['state_dict'])
        net.eval()
        print(f"  checkpoint found -> Results/models/{name}.pt  (training skipped)")
        return net, ckpt.get('history', [])
    t0 = time.time()
    net, history = train_fn()
    if SMOKE_MODE:
        # A smoke run trains on shortened schedules, so its weights must never
        # overwrite the shipped checkpoints.
        print(f"  trained in {time.time()-t0:.1f}s  (smoke run, checkpoint not written)")
    else:
        torch.save(dict(state_dict=net.state_dict(), history=history, meta=meta or {}), path)
        print(f"  trained in {time.time()-t0:.1f}s  ->  Results/models/{name}.pt")
    return net, history


def count_params(net):
    return sum(p.numel() for p in net.parameters())

## 3. Analytic surface library**Ridge.** $f(p) = z - a\exp(-b\,y'^2)$ with $y' = -\sin\phi\,x + \cos\phi\,y$rotating the ridge in the $xy$-plane. The linearised signed distance is$f / \|\nabla f\|$ with $\|\nabla f\|^2 = 1 + 4a^2b^2y'^2\exp(-2by'^2)$.**Saddle.** $f(p) = z - a(x^2 - y^2)$, a smooth surface with no localisedfeature, used as the null-result control.Analytic ground truth gives exact signed-distance and normal references, so thepaired evaluation is unambiguous.

In [ ]:
def make_ridge_sdf(a=0.5, b=5.0, phi=0.0, centre=(0.0, 0.0, 0.0)):
    cos_p = math.cos(phi); sin_p = math.sin(phi)
    cx, cy, cz = centre
    def sdf(p):
        x = p[..., 0] - cx
        y = p[..., 1] - cy
        z = p[..., 2] - cz
        y_prime = -sin_p * x + cos_p * y
        gauss = a * torch.exp(-b * y_prime ** 2)
        f = z - gauss
        # Analytical gradient norm
        df_dy_prime = 2 * a * b * y_prime * torch.exp(-b * y_prime ** 2)
        df_dx = df_dy_prime * (-sin_p)
        df_dy = df_dy_prime *   cos_p
        df_dz = torch.ones_like(f)
        gnorm = torch.sqrt(df_dx ** 2 + df_dy ** 2 + df_dz ** 2)
        return f / gnorm
    return sdf


def make_saddle_sdf(a=0.4, centre=(0.0, 0.0, 0.0)):
    cx, cy, cz = centre
    def sdf(p):
        x = p[..., 0] - cx
        y = p[..., 1] - cy
        z = p[..., 2] - cz
        f = z - a * (x ** 2 - y ** 2)
        df_dx = -2 * a * x
        df_dy =  2 * a * y
        df_dz = torch.ones_like(f)
        gnorm = torch.sqrt(df_dx ** 2 + df_dy ** 2 + df_dz ** 2)
        return f / gnorm
    return sdf


# Sanity check
sdf_test = make_ridge_sdf(a=0.5, b=5.0, phi=0.0)
test_pts = torch.tensor([
    [0.0, 0.0, 0.5],   # on the ridge crest -> sdf = 0
    [0.0, 0.0, 0.0],   # below the crest    -> sdf < 0
    [0.0, 0.0, 1.0],   # above the crest    -> sdf > 0
    [0.0, 1.0, 0.0],   # on the flat far away -> sdf ~= 0
], dtype=DTYPE)
print("Ridge SDF sanity check (a=0.5, b=5, phi=0):")
print(f"  on-crest (0, 0, 0.5):       sdf = {sdf_test(test_pts[0]).item():+.4f}  (expected 0)")
print(f"  below-crest (0, 0, 0.0):    sdf = {sdf_test(test_pts[1]).item():+.4f}  (expected -ve, |.|>0)")
print(f"  above-crest (0, 0, 1.0):    sdf = {sdf_test(test_pts[2]).item():+.4f}  (expected +ve)")
print(f"  far-flat   (0, 1, 0.0):     sdf = {sdf_test(test_pts[3]).item():+.4f}  (expected ~0)")

## 4. Mesh and dataset generatorsEach instance is a nine-by-nine jittered grid lifted onto the analytic surface.Interior vertices are jittered so the network sees varied stencil geometriesrather than one regular grid, and boundary vertices are left clean. Ridgeparameters are drawn with $a \in [0.3, 0.8]$, $b \in [3, 10]$, a random rotation$\phi$, and a small centre offset. The ridge family is split twenty to ten intotraining and held-out test sets.

In [ ]:
def grid_patch(nx=9, ny=9, half_size=1.0, jitter=0.04, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.linspace(-half_size, half_size, nx, dtype=DTYPE)
    y = torch.linspace(-half_size, half_size, ny, dtype=DTYPE)
    X, Y = torch.meshgrid(x, y, indexing='ij')
    Vxy = torch.stack([X.flatten(), Y.flatten()], dim=1)
    if jitter > 0:
        # Don't jitter boundary nodes (clean boundary)
        is_interior = ((Vxy[:, 0].abs() < half_size - 1e-6) &
                       (Vxy[:, 1].abs() < half_size - 1e-6))
        noise = (torch.rand(Vxy.shape, generator=g, dtype=DTYPE) - 0.5) * jitter * 2 * half_size / max(nx, ny)
        Vxy = Vxy + noise * is_interior.unsqueeze(1)
    F_list = []
    for i in range(nx-1):
        for j in range(ny-1):
            v00 = i*ny + j; v10 = (i+1)*ny + j
            v01 = i*ny + j+1; v11 = (i+1)*ny + j+1
            F_list.append([v00, v10, v11]); F_list.append([v00, v11, v01])
    return Vxy, torch.tensor(F_list, dtype=torch.long)


def lift_to_ridge(Vxy, a, b, phi, centre=(0.0, 0.0, 0.0)):
    cx, cy, cz = centre
    cos_p, sin_p = math.cos(phi), math.sin(phi)
    x = Vxy[:, 0] - cx; y = Vxy[:, 1] - cy
    y_prime = -sin_p * x + cos_p * y
    z = a * torch.exp(-b * y_prime ** 2) + cz
    return torch.stack([Vxy[:, 0], Vxy[:, 1], z], dim=1)


def lift_to_saddle(Vxy, a, centre=(0.0, 0.0, 0.0)):
    cx, cy, cz = centre
    x = Vxy[:, 0] - cx; y = Vxy[:, 1] - cy
    z = a * (x ** 2 - y ** 2) + cz
    return torch.stack([Vxy[:, 0], Vxy[:, 1], z], dim=1)


def make_ridge_dataset(n_instances, seed_base=0):
    """Generate (mesh, sdf_fn, params) tuples."""
    samples = []
    rng = np.random.default_rng(seed_base)
    for i in range(n_instances):
        a   = float(rng.uniform(0.3, 0.8))
        b   = float(rng.uniform(3.0, 10.0))
        phi = float(rng.uniform(0.0, math.pi))
        cx  = float(rng.uniform(-0.1, 0.1))
        cy  = float(rng.uniform(-0.1, 0.1))
        cz  = 0.0
        Vxy, F = grid_patch(nx=9, ny=9, half_size=1.0, jitter=0.04, seed=seed_base + i)
        V = lift_to_ridge(Vxy, a=a, b=b, phi=phi, centre=(cx, cy, cz))
        m = Mesh(V, F)
        sdf_fn = make_ridge_sdf(a=a, b=b, phi=phi, centre=(cx, cy, cz))
        samples.append(dict(
            mesh=m, sdf=sdf_fn,
            params=dict(a=a, b=b, phi=phi, cx=cx, cy=cy),
            name=f'ridge_{i}_a{a:.2f}_b{b:.1f}_phi{phi:.2f}'
        ))
    return samples


def make_saddle_dataset(n_instances, seed_base=10000):
    samples = []
    rng = np.random.default_rng(seed_base)
    for i in range(n_instances):
        a  = float(rng.uniform(0.3, 0.6))
        cx = float(rng.uniform(-0.1, 0.1))
        cy = float(rng.uniform(-0.1, 0.1))
        Vxy, F = grid_patch(nx=9, ny=9, half_size=1.0, jitter=0.04, seed=seed_base + i)
        V = lift_to_saddle(Vxy, a=a, centre=(cx, cy, 0.0))
        m = Mesh(V, F)
        sdf_fn = make_saddle_sdf(a=a, centre=(cx, cy, 0.0))
        samples.append(dict(mesh=m, sdf=sdf_fn,
                             params=dict(a=a, cx=cx, cy=cy), name=f'saddle_{i}_a{a:.2f}'))
    return samples


# Build datasets. SMOKE_MODE shrinks every split for a quick functional check.
N_RIDGE_TRAIN, N_RIDGE_TEST = (4, 3) if SMOKE_MODE else (20, 10)
N_SADDLE_TRAIN, N_SADDLE_TEST = (3, 2) if SMOKE_MODE else (12, 4)

RIDGE_TRAIN  = make_ridge_dataset(N_RIDGE_TRAIN, seed_base=42)
RIDGE_TEST   = make_ridge_dataset(N_RIDGE_TEST,  seed_base=999)
SADDLE_TRAIN = make_saddle_dataset(N_SADDLE_TRAIN, seed_base=10000)
SADDLE_TEST  = make_saddle_dataset(N_SADDLE_TEST,  seed_base=11000)

print(f"Ridge   train: {len(RIDGE_TRAIN)} instances    test: {len(RIDGE_TEST)} instances")
print(f"Saddle  train: {len(SADDLE_TRAIN)} instances    test: {len(SADDLE_TEST)} instances")
print(f"Each mesh has {RIDGE_TRAIN[0]['mesh'].n_v} vertices, {RIDGE_TRAIN[0]['mesh'].n_f} faces")

## 5. Evaluation metricsMetrics are reported in two regions. The near-feature region is $|y'| < 0.3$,where the ridge carries appreciable curvature and Loop's truncation errorconcentrates. The global region is every inserted vertex, including the flatsurround. Paired comparisons use bootstrap confidence intervals over twothousand resamples of the held-out instances.

In [ ]:
@torch.no_grad()
def evaluate_one(mesh, sdf_fn, q_t, q_0, params=None,
                 use_h2=True, e_len=None, edges=None, edge_faces=None,
                 C_for_ratio=None):
    """Compute all metrics on a single mesh given the inserted vertices q_t."""
    if edges is None:
        edges, edge_faces, _ = build_edge_data(mesh.F)
    if e_len is None:
        _, e_len = edge_frames(mesh.V, mesh.F, edges, edge_faces)

    # Near-feature mask: based on y' coordinate of the inserted vertex
    near_mask = None
    if params is not None and 'phi' in params:
        cos_p, sin_p = math.cos(params['phi']), math.sin(params['phi'])
        cx = params.get('cx', 0.0); cy = params.get('cy', 0.0)
        x = q_t[:, 0] - cx; y = q_t[:, 1] - cy
        y_prime = -sin_p * x + cos_p * y
        near_mask = (y_prime.abs() < 0.3)
    elif params is not None and ('cx' in params and 'cy' in params and 'a' in params and 'phi' not in params):
        # Saddle: near origin = high curvature
        x = q_t[:, 0] - params['cx']; y = q_t[:, 1] - params['cy']
        near_mask = ((x ** 2 + y ** 2) < 0.3 ** 2)

    n_t = discrete_normal_at_inserts(mesh.V, mesh.F, q_t, edges, edge_faces)
    with torch.enable_grad():
        n_target = surface_normal(sdf_fn, q_t.detach())

    d_vals = sdf_fn(q_t)
    metrics = {}
    metrics['sdf_rmse']  = float(torch.sqrt((d_vals ** 2).mean()))
    metrics['hausdorff'] = float(d_vals.abs().max())
    cosang = (n_t * n_target).sum(dim=1).clamp(-1, 1)
    metrics['normal_err'] = float(torch.acos(cosang).mean())
    if near_mask is not None and near_mask.any():
        metrics['sdf_rmse_near']  = float(torch.sqrt((d_vals[near_mask] ** 2).mean()))
        metrics['hausdorff_near'] = float(d_vals[near_mask].abs().max())
        metrics['normal_err_near'] = float(torch.acos(cosang[near_mask]).mean())
    else:
        metrics['sdf_rmse_near']  = metrics['sdf_rmse']
        metrics['hausdorff_near'] = metrics['hausdorff']
        metrics['normal_err_near'] = metrics['normal_err']

    h2 = (e_len ** 2).clamp_min(1e-12)
    if C_for_ratio is not None:
        metrics['prox_ratio_max'] = float(((q_t - q_0).norm(dim=1) / (C_for_ratio * h2)).max())
        metrics['prox_ratio_mean'] = float(((q_t - q_0).norm(dim=1) / (C_for_ratio * h2)).mean())
    return metrics


@torch.no_grad()
def evaluate_loop_pair(samples):
    rows = []
    for s in samples:
        m = s['mesh']
        edges, edge_faces, opps = build_edge_data(m.F)
        q_0 = loop_edge_vertices(m.V, edges, opps)
        rows.append(evaluate_one(m, s['sdf'], q_0, q_0, params=s.get('params'),
                                  edges=edges, edge_faces=edge_faces, C_for_ratio=1.0))
    return rows


@torch.no_grad()
def evaluate_constrained_pair(samples, net, C, c_gate=C_GATE):
    rows = []
    for s in samples:
        m = s['mesh']
        q_t, q_0, aux = constrained_edge_vertices(m, net, c_gate=c_gate, return_aux=True)
        rows.append(evaluate_one(m, s['sdf'], q_t, q_0, params=s.get('params'),
                                  edges=aux['edges'], edge_faces=aux['edge_faces'],
                                  e_len=aux['e_len'], C_for_ratio=C))
    return rows


@torch.no_grad()
def evaluate_unconstrained_pair(samples, net, C_for_ratio=0.5):
    rows = []
    for s in samples:
        m = s['mesh']
        q_t, q_0 = net(m)
        edges, edge_faces, _ = build_edge_data(m.F)
        _, e_len = edge_frames(m.V, m.F, edges, edge_faces)
        rows.append(evaluate_one(m, s['sdf'], q_t, q_0, params=s.get('params'),
                                  edges=edges, edge_faces=edge_faces, e_len=e_len,
                                  C_for_ratio=C_for_ratio))
    return rows


def paired_summary(rows_a, rows_b, key, n_bootstrap=2000, ci=95):
    """Bootstrap CI on the paired relative improvement (Loop minus operator).
    Uses the bootstrap distribution itself for both central estimate and CI,
    so the CI brackets the central estimate by construction."""
    a = np.array([r[key] for r in rows_a])
    b = np.array([r[key] for r in rows_b])
    rng = np.random.default_rng(0)
    n = len(a)
    diffs = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        a_mean = np.mean(a[idx])
        if a_mean > 0:
            diffs.append((a_mean - np.mean(b[idx])) / a_mean * 100)
        else:
            diffs.append(0.0)
    diffs = np.array(diffs)
    lo, hi = np.percentile(diffs, [(100-ci)/2, 100 - (100-ci)/2])
    rel_imp = float(np.mean(diffs))   # bootstrap mean of the paired-ratio metric
    return dict(mean_a=float(np.mean(a)), mean_b=float(np.mean(b)),
                rel_improvement_pct=rel_imp,
                ci_lo_pct=float(lo), ci_hi_pct=float(hi))

## 6. Structural certificationThe properties below are consequences of the parameterisation, so the honest wayto test them is to look for violations rather than to confirm them on aconvenient example. Each certificate is evaluated across a battery of randomweight draws and shape-regular meshes, and then attacked directly by gradientascent on the quantity the theorem bounds.Bounded displacement caps the per-edge correction at $C h_i^2$. Equivariancemeans the operator commutes with rigid motion. Affine reproduction means theoperator coincides with Loop on planar input. Because these are exactstatements, they are measured in double precision, and the residuals reportedare floating-point noise rather than modelling error.

In [ ]:
import contextlib


@contextlib.contextmanager
def precision(dtype):
    """Temporarily switch the global working precision.

    The architectural certificates are exact statements, so they are measured in
    double precision and the residuals reported below are floating-point noise
    rather than modelling error. Everything else in the notebook runs in single
    precision, which is what the operator uses in practice.
    """
    global DTYPE
    old = DTYPE
    DTYPE = dtype
    torch.set_default_dtype(dtype)
    try:
        yield
    finally:
        DTYPE = old
        torch.set_default_dtype(old)


# ---- Certification test meshes -----------------------------------------

def lift_to_paraboloid(Vxy, a=0.35):
    x, y = Vxy[:, 0], Vxy[:, 1]
    return torch.stack([x, y, a * (x ** 2 + y ** 2)], dim=1)


def _icosahedron():
    t = (1.0 + math.sqrt(5.0)) / 2.0
    V = torch.tensor([[-1, t, 0], [1, t, 0], [-1, -t, 0], [1, -t, 0],
                      [0, -1, t], [0, 1, t], [0, -1, -t], [0, 1, -t],
                      [t, 0, -1], [t, 0, 1], [-t, 0, -1], [-t, 0, 1]], dtype=DTYPE)
    F = torch.tensor([[0, 11, 5], [0, 5, 1], [0, 1, 7], [0, 7, 10], [0, 10, 11],
                      [1, 5, 9], [5, 11, 4], [11, 10, 2], [10, 7, 6], [7, 1, 8],
                      [3, 9, 4], [3, 4, 2], [3, 2, 6], [3, 6, 8], [3, 8, 9],
                      [4, 9, 5], [2, 4, 11], [6, 2, 10], [8, 6, 7], [9, 8, 1]],
                     dtype=torch.long)
    return Mesh(V / V.norm(dim=1, keepdim=True), F)


def icosphere(n_split=2):
    """Shape-regular closed triangle mesh built by 1-to-4 splitting with radial projection."""
    m = _icosahedron()
    for _ in range(n_split):
        V, F = m.V, m.F
        verts = [v for v in V]
        cache = {}

        def midpoint(a, b):
            key = (min(a, b), max(a, b))
            if key not in cache:
                p = 0.5 * (V[a] + V[b])
                verts.append(p / p.norm())
                cache[key] = len(verts) - 1
            return cache[key]

        new_F = []
        for a, b, c in F.tolist():
            ab, bc, ca = midpoint(a, b), midpoint(b, c), midpoint(c, a)
            new_F += [[a, ab, ca], [b, bc, ab], [c, ca, bc], [ab, bc, ca]]
        m = Mesh(torch.stack(verts), torch.tensor(new_F, dtype=torch.long))
    return m


def bumpy_sphere(n_split=2, amp=0.12, k_lobes=4):
    """Closed mesh with a non-trivial curvature field, so the gate is active."""
    m = icosphere(n_split)
    u = m.V / m.V.norm(dim=1, keepdim=True)
    bump = 1.0 + amp * torch.sin(k_lobes * torch.atan2(u[:, 1], u[:, 0])) * (1.0 - u[:, 2] ** 2)
    return Mesh(u * bump.unsqueeze(1), m.F)


def planar_patch(nx=9, jitter=0.05, seed=7):
    """Exactly planar mesh: all face normals coincide, so the curvature feature reads zero."""
    Vxy, F = grid_patch(nx=nx, ny=nx, half_size=1.0, jitter=jitter, seed=seed)
    V = torch.stack([Vxy[:, 0], Vxy[:, 1], torch.zeros_like(Vxy[:, 0])], dim=1)
    return Mesh(V, F)


def paraboloid_patch(nx=9, jitter=0.05, seed=3, a=0.35):
    Vxy, F = grid_patch(nx=nx, ny=nx, half_size=1.0, jitter=jitter, seed=seed)
    return Mesh(lift_to_paraboloid(Vxy, a=a), F)


def certification_meshes():
    return {'paraboloid': paraboloid_patch(),
            'bumpy sphere': bumpy_sphere(),
            'planar': planar_patch()}


# ---- Individual certificates -------------------------------------------

def random_rigid_motion(seed=0):
    """Draw (R, t) in SE(3) with R distributed according to the Haar measure on SO(3)."""
    g = torch.Generator().manual_seed(seed)
    A = torch.randn(3, 3, generator=g, dtype=DTYPE)
    Q, R = torch.linalg.qr(A)
    Q = Q * torch.sign(torch.diagonal(R)).unsqueeze(0)
    if torch.det(Q) < 0:
        Q = Q.clone()
        Q[:, 0] = -Q[:, 0]
    t = torch.randn(3, generator=g, dtype=DTYPE) * 2.0
    return Q, t


@torch.no_grad()
def certify_bounded_displacement(mesh, net, C):
    """Theorem 1, reported as the ratio of realised displacement to the cap C h_i^2."""
    q_t, q_0, aux = constrained_edge_vertices(mesh, net, c_gate=C_GATE, return_aux=True)
    h2 = (aux['e_len'] ** 2).clamp_min(1e-14)
    return float(((q_t - q_0).norm(dim=1) / (C * h2)).max())


@torch.no_grad()
def certify_equivariance(mesh, net, n_trials=8):
    """Theorem 2, the largest discrepancy between S_theta(gM) and g S_theta(M)."""
    q_ref, _ = constrained_edge_vertices(mesh, net, c_gate=C_GATE)
    worst = 0.0
    for trial in range(n_trials):
        R, t = random_rigid_motion(seed=trial)
        moved = Mesh(mesh.V @ R.T + t, mesh.F)
        q_moved, _ = constrained_edge_vertices(moved, net, c_gate=C_GATE)
        worst = max(worst, float((q_moved - (q_ref @ R.T + t)).norm(dim=1).max()))
    return worst


@torch.no_grad()
def certify_affine_reproduction(net, mesh=None):
    """Theorem 3, the departure from Loop on exactly planar input."""
    m = mesh if mesh is not None else planar_patch()
    q_t, q_0 = constrained_edge_vertices(m, net, c_gate=C_GATE)
    return float((q_t - q_0).norm(dim=1).max())


def adversarial_proximity_search(C=0.5, n_steps=200, lr=5e-2, seed=0):
    """Maximise the proximity ratio jointly over network weights and mesh geometry.

    Theorem 1 is claimed for any finite weights on any shape-regular mesh, so a
    search free in both arguments should be unable to push the ratio past one.
    """
    torch.manual_seed(seed)
    net = CorrectionNet(dim_in=5, hidden=32, n_layers=2, C=C)
    base = paraboloid_patch()
    dV = torch.zeros_like(base.V, requires_grad=True)
    opt = torch.optim.Adam(list(net.parameters()) + [dV], lr=lr)
    worst = 0.0
    for _ in range(n_steps):
        opt.zero_grad()
        m = Mesh(base.V + 0.15 * torch.tanh(dV), base.F)
        q_t, q_0, aux = constrained_edge_vertices(m, net, c_gate=C_GATE, return_aux=True)
        h2 = (aux['e_len'] ** 2).clamp_min(1e-14)
        ratio = (q_t - q_0).norm(dim=1) / (C * h2)
        worst = max(worst, float(ratio.max()))
        (-ratio.max()).backward()
        opt.step()
    return worst


def adversarial_equivariance_search(C=0.5, n_steps=150, lr=5e-2, seed=1):
    """Maximise the equivariance residual over network weights."""
    torch.manual_seed(seed)
    net = CorrectionNet(dim_in=5, hidden=32, n_layers=2, C=C)
    m = paraboloid_patch()
    R, t = random_rigid_motion(seed=5)
    moved = Mesh(m.V @ R.T + t, m.F)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    worst = 0.0
    for _ in range(n_steps):
        opt.zero_grad()
        q_ref, _ = constrained_edge_vertices(m, net, c_gate=C_GATE)
        q_mov, _ = constrained_edge_vertices(moved, net, c_gate=C_GATE)
        err = (q_mov - (q_ref @ R.T + t)).norm(dim=1).max()
        worst = max(worst, float(err))
        (-err).backward()
        opt.step()
    return worst


def adversarial_affine_search(C=0.5, n_steps=150, lr=5e-2, seed=2):
    """Maximise the departure from Loop on exactly planar input over network weights."""
    torch.manual_seed(seed)
    net = CorrectionNet(dim_in=5, hidden=32, n_layers=2, C=C)
    m = planar_patch()
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    worst = 0.0
    for _ in range(n_steps):
        opt.zero_grad()
        q_t, q_0 = constrained_edge_vertices(m, net, c_gate=C_GATE)
        err = (q_t - q_0).norm(dim=1).max()
        worst = max(worst, float(err))
        if err.requires_grad and err.grad_fn is not None:
            (-err).backward()
            opt.step()
        else:
            break
    return worst

### Certificates and adversarial stress tests

In [ ]:
print("=" * 78)
print("STRUCTURAL CERTIFICATION  --  properties that hold for any finite weights")
print("=" * 78)

CERT_C = 0.5
cert_records = []

with precision(torch.float64):
    meshes = certification_meshes()
    nets = {}
    for s in range(5):
        torch.manual_seed(100 + s)
        nets[f'random init {s}'] = CorrectionNet(dim_in=5, hidden=32, n_layers=2, C=CERT_C)

    worst_prox, worst_equi, worst_affine = 0.0, 0.0, 0.0
    for net_name, net in nets.items():
        for mesh_name, mesh in meshes.items():
            worst_prox = max(worst_prox, certify_bounded_displacement(mesh, net, CERT_C))
            worst_equi = max(worst_equi, certify_equivariance(mesh, net))
        worst_affine = max(worst_affine, certify_affine_reproduction(net))

    print(f"\n  Battery: {len(nets)} random weight draws x {len(meshes)} shape-regular meshes")
    print(f"    max proximity ratio      = {worst_prox:.6f}   (architectural cap = 1)")
    print(f"    max equivariance error   = {worst_equi:.3e}   (expected 0)")
    print(f"    max affine repro. error  = {worst_affine:.3e}   (expected 0)")

    print("\n  Adversarial searches (gradient ascent against each bound)")
    adv_prox = adversarial_proximity_search(C=CERT_C, n_steps=steps(200))
    adv_equi = adversarial_equivariance_search(C=CERT_C, n_steps=steps(150))
    adv_aff = adversarial_affine_search(C=CERT_C, n_steps=steps(150))
    print(f"    proximity ratio under attack        = {adv_prox:.6f}   (cap = 1)")
    print(f"    equivariance error under attack     = {adv_equi:.3e}   (expected 0)")
    print(f"    affine repro. error under attack    = {adv_aff:.3e}   (expected 0)")

cert_records = [
    dict(test='bounded displacement (T1)', quantity='max proximity ratio',
         expected='<= 1', measured=f'{worst_prox:.6f}'),
    dict(test='SE(3)-equivariance (T2)', quantity='max positional discrepancy',
         expected='0', measured=f'{worst_equi:.3e}'),
    dict(test='affine reproduction (T3)', quantity='max departure from Loop on planar input',
         expected='0', measured=f'{worst_affine:.3e}'),
    dict(test='bounded displacement, adversarial', quantity='max proximity ratio',
         expected='<= 1', measured=f'{adv_prox:.6f}'),
    dict(test='equivariance, adversarial', quantity='max positional discrepancy',
         expected='0', measured=f'{adv_equi:.3e}'),
    dict(test='affine reproduction, adversarial', quantity='max departure from Loop',
         expected='0', measured=f'{adv_aff:.3e}'),
]
write_table('structural_certification', ['test', 'quantity', 'expected', 'measured'], cert_records)

RESULTS['certification'] = dict(
    max_proximity_ratio=worst_prox, max_equivariance_error=worst_equi,
    max_affine_error=worst_affine, adversarial_proximity_ratio=adv_prox,
    adversarial_equivariance_error=adv_equi, adversarial_affine_error=adv_aff,
    n_weight_draws=len(nets), meshes=list(meshes.keys()), C=CERT_C)

## 7. Spectral certificationNear a valence-$k$ vertex, one subdivision step is a linear map on the$(k+1)$-dimensional space of star configurations. Reif's conditions require asingle subdominant eigenvalue, a complex-conjugate pair of tangent eigenvalues$\lambda_t = |\lambda_t|e^{\pm i 2\pi/k}$, and a strictly larger gap to theremaining spectrum. For Loop, $|\lambda_t| = \tfrac{3}{8} + \tfrac{1}{4}\cos(2\pi/k)$.Two claims are checked here. First, that the local subdivision matrix extractedfrom a planar valence-$k$ star reproduces Loop's analytic tangent eigenvalue for$k = 3, \ldots, 12$. Second, that the linearisation of $S_\theta$ at the planarstar equals Loop's, which is exactly the spectral inheritance theorem. The gatehas a zero of order two at the planar reference, so both the correction and itsJacobian vanish there.Off the planar reference no exact statement is available, so the margin iscertified numerically. Lifting the centre vertex by $\delta$ and recomputing thespectrum shows the gap preserved at $\delta = 0$ for every budget, degradingsmoothly with $\delta$ at a rate set by $C$. This is the sense in which $C$ actsas a spectral safety parameter alongside its role as a capacity parameter.

In [ ]:
def valence_k_star(k, radius=1.0, centre_lift=0.0):
    """Closed fan of k triangles around a single interior vertex of valence k.

    The k spoke edges are interior and therefore carry the learned correction. The
    outer ring edges are boundary edges and are left to Loop's boundary rule. With
    centre_lift equal to zero the star is exactly planar, which is the reference
    configuration of the spectral inheritance theorem.
    """
    ang = torch.arange(k, dtype=DTYPE) * (2.0 * math.pi / k)
    ring = torch.stack([radius * torch.cos(ang),
                        radius * torch.sin(ang),
                        torch.zeros(k, dtype=DTYPE)], dim=1)
    centre = torch.tensor([[0.0, 0.0, centre_lift]], dtype=DTYPE)
    V = torch.cat([centre, ring], dim=0)
    F = torch.tensor([[0, 1 + i, 1 + ((i + 1) % k)] for i in range(k)], dtype=torch.long)
    return Mesh(V, F)


def star_map_loop(V, F):
    """One Loop step restricted to the star: returns [new centre; new spoke vertices]."""
    m = Mesh(V, F)
    new_centre = loop_old_vertex_update(V, F)[0]
    edges, edge_faces, opps = build_edge_data(F)
    q0 = loop_edge_vertices(V, edges, opps)
    return torch.cat([new_centre.unsqueeze(0), q0], dim=0)


def star_map_pns(V, F, net):
    """One PNS step restricted to the star, in the same ordering as star_map_loop."""
    m = Mesh(V, F)
    new_centre = loop_old_vertex_update(V, F)[0]
    q_t, _ = constrained_edge_vertices(m, net, c_gate=C_GATE)
    return torch.cat([new_centre.unsqueeze(0), q_t], dim=0)


def star_jacobian(fn, V):
    """Full linearisation of a star map, shaped (k+1, 3, k+1, 3)."""
    return torch.autograd.functional.jacobian(fn, V, vectorize=False)


def subdivision_matrix(J):
    """Scalar (k+1) x (k+1) subdivision matrix read off the normal-direction block."""
    return J[:, 2, :, 2]


def reif_quantities(S):
    """Return the sorted eigenvalue moduli, the tangent modulus, and the Reif gap.

    The dominant eigenvalue is one, the tangent eigenvalues form a complex conjugate
    pair of equal modulus, and the gap is the distance from that modulus to the
    largest remaining eigenvalue modulus.
    """
    ev = torch.linalg.eigvals(S.to(torch.complex128))
    mods = torch.sort(ev.abs(), descending=True).values.real
    lam_t = float(mods[1]) if mods.numel() > 1 else float('nan')
    lam_next = float(mods[3]) if mods.numel() > 3 else float('nan')
    return mods, lam_t, (lam_t - lam_next)


def loop_tangent_analytic(k):
    return 3.0 / 8.0 + 0.25 * math.cos(2.0 * math.pi / k)

### Valence sweep and off-planar margin

In [ ]:
print("=" * 78)
print("SPECTRAL CERTIFICATION  --  planar inheritance and off-planar margin")
print("=" * 78)

VALENCES = list(range(3, 13))
DELTAS = [0.0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4]
C_SPECTRAL = [0.1, 0.25, 0.5, 1.0]

spectral_rows = []
offplanar = {C: {'delta': [], 'gap': [], 'lam_t': []} for C in C_SPECTRAL}

with precision(torch.float64):
    torch.manual_seed(11)
    net_probe = CorrectionNet(dim_in=5, hidden=32, n_layers=2, C=0.5)

    print(f"\n{'k':>3s}  {'|lambda_t| Loop':>16s}  {'analytic':>10s}  {'Reif gap':>9s}  {'max |DS_theta - DS_0|':>22s}")
    print('-' * 72)
    for k in VALENCES:
        star = valence_k_star(k)
        J_loop = star_jacobian(lambda V: star_map_loop(V, star.F), star.V)
        J_pns = star_jacobian(lambda V: star_map_pns(V, star.F, net_probe), star.V)
        mods, lam_t, gap = reif_quantities(subdivision_matrix(J_loop))
        dev = float((J_pns - J_loop).abs().max())
        print(f"{k:3d}  {lam_t:16.9f}  {loop_tangent_analytic(k):10.6f}  {gap:9.4f}  {dev:22.3e}")
        spectral_rows.append(dict(valence=k, lambda_t_measured=f'{lam_t:.9f}',
                                  lambda_t_analytic=f'{loop_tangent_analytic(k):.9f}',
                                  reif_gap=f'{gap:.6f}',
                                  max_linearisation_deviation=f'{dev:.3e}'))

    # Off-planar behaviour on the valence-six star.
    for C in C_SPECTRAL:
        torch.manual_seed(11)
        net_C = CorrectionNet(dim_in=5, hidden=32, n_layers=2, C=C)
        for d in DELTAS:
            star = valence_k_star(6, centre_lift=d)
            J = star_jacobian(lambda V: star_map_pns(V, star.F, net_C), star.V)
            _, lam_t, gap = reif_quantities(subdivision_matrix(J))
            offplanar[C]['delta'].append(d)
            offplanar[C]['gap'].append(gap)
            offplanar[C]['lam_t'].append(lam_t)

write_table('spectral_certification',
            ['valence', 'lambda_t_measured', 'lambda_t_analytic', 'reif_gap',
             'max_linearisation_deviation'], spectral_rows)

print("\n  At the planar reference the two linearisations agree to machine precision for")
print("  every valence tested, so PNS inherits Loop's tangent eigenvalues and Reif gap.")

# ---- Figure: spectral inheritance and off-planar margin ----
fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.0), constrained_layout=True)

ax = axes[0]
ax.plot(VALENCES, [loop_tangent_analytic(k) for k in VALENCES], '-', color='k',
        lw=1.4, label=r'analytic $\frac{3}{8} + \frac{1}{4}\cos(2\pi/k)$')
ax.plot(VALENCES, [float(r['lambda_t_measured']) for r in spectral_rows], 'o',
        color=PALETTE[2], markersize=8, label=r'measured from $DS_\theta$')
ax.set_xlabel(r'Valence $k$')
ax.set_ylabel(r'Tangent eigenvalue modulus $|\lambda_t|$')
ax.set_title('Planar spectral inheritance')
ax.set_xticks(VALENCES)
ax.legend(loc='lower right')

ax = axes[1]
for C, colour in zip(C_SPECTRAL, [PALETTE[0], PALETTE[2], PALETTE[1], PALETTE[3]]):
    ax.plot(offplanar[C]['delta'], offplanar[C]['gap'], 'o-', color=colour,
            markersize=6, label=f'$C = {C}$')
ax.set_xlabel(r'Centre lift $\delta$ off the planar reference')
ax.set_ylabel(r'Reif spectral gap')
ax.set_title(r'Off-planar margin narrows in proportion to $C$')
ax.legend(loc='lower left', ncol=2)

fig.savefig(OUT / 'spectral_inheritance.png', dpi=180, bbox_inches='tight')
plt.show()

RESULTS['spectral'] = dict(per_valence=spectral_rows,
                           off_planar={str(C): offplanar[C] for C in C_SPECTRAL})

## 8. Architectural ablationEach architectural element is removed in turn and the resulting operator ismeasured on the same benchmark. The purpose is to identify which property eachelement protects, not to find a better variant. The random-weight row keeps thefull architecture and replaces trained weights by random initialisation, whichseparates what the architecture guarantees from what training contributes.

In [ ]:
def make_subdivider(net, use_gate=True, use_frame=True, use_h2=True):
    """Generalisation of subdivide_constrained with the architectural switches exposed.

    Interior edges receive the learned correction under the requested switch setting,
    and boundary edges keep Loop's boundary midpoint, so the refined topology matches
    subdivide_loop exactly and the operators remain comparable level by level.
    """
    def op(mesh):
        edge_keys, edge_to_id, edge_opps = _build_full_edge_index(mesh.F)
        new_edge_v = _new_edge_vertices_loop(mesh.V, edge_keys, edge_opps)
        edges_int, _, _ = build_edge_data(mesh.F)
        with torch.no_grad():
            q_theta, _ = constrained_edge_vertices(
                mesh, net, c_gate=C_GATE,
                use_gate=use_gate, use_frame=use_frame, use_h2=use_h2)
        for ei in range(edges_int.shape[0]):
            a, b = int(edges_int[ei, 0]), int(edges_int[ei, 1])
            key = (a, b) if a < b else (b, a)
            new_edge_v[edge_to_id[key]] = q_theta[ei]
        return _topological_subdivide(mesh, new_edge_v)
    return op


@torch.no_grad()
def ablation_one_step(samples, net, C, use_gate=True, use_frame=True, use_h2=True):
    """One-step metrics for a switch setting, including the realised proximity ratio."""
    rows = []
    for s in samples:
        m = s['mesh']
        q_t, q_0, aux = constrained_edge_vertices(
            m, net, c_gate=C_GATE, return_aux=True,
            use_gate=use_gate, use_frame=use_frame, use_h2=use_h2)
        rows.append(evaluate_one(m, s['sdf'], q_t, q_0, params=s.get('params'),
                                 edges=aux['edges'], edge_faces=aux['edge_faces'],
                                 e_len=aux['e_len'], C_for_ratio=C))
    return rows


@torch.no_grad()
def repeated_summary(sample, op_fn, n_levels=3, C_arch=0.5):
    """Max proximity ratio and max face-normal jump after n_levels repeated steps."""
    m = sample['mesh']
    worst_ratio = 0.0
    for _ in range(n_levels):
        parent = m
        edge_keys, _, edge_opps = _build_full_edge_index(parent.F)
        q_0_full = _new_edge_vertices_loop(parent.V, edge_keys, edge_opps)
        e_len_full = torch.stack([
            (parent.V[a] - parent.V[b]).norm() for a, b in edge_keys])
        h2 = (e_len_full ** 2).clamp_min(1e-12)
        m = op_fn(parent)
        q_t_full = m.V[parent.n_v:]
        worst_ratio = max(worst_ratio,
                          float(((q_t_full - q_0_full).norm(dim=1) / (C_arch * h2)).max()))
    edges, edge_faces, _ = build_edge_data(m.F)
    Fn = _face_normals(m.V, m.F)
    cos_dih = (Fn[edge_faces[:, 0]] * Fn[edge_faces[:, 1]]).sum(dim=1).clamp(-1, 1)
    return worst_ratio, float(torch.acos(cos_dih).max())


def _build_ablation_net(sw, C):
    net = CorrectionNet(dim_in=5, hidden=32, n_layers=2, C=C)
    net.bounded = sw['bounded']
    return net


@torch.no_grad()
def certify_affine_reproduction_switched(net, sw, mesh=None):
    """Affine reproduction under a given switch setting, used by the ablation table."""
    m = mesh if mesh is not None else planar_patch()
    q_t, q_0 = constrained_edge_vertices(
        m, net, c_gate=C_GATE,
        use_gate=sw['use_gate'], use_frame=sw['use_frame'], use_h2=sw['use_h2'])
    return float((q_t - q_0).norm(dim=1).max())

### Ablation table

In [ ]:
print("=" * 78)
print("ARCHITECTURAL ABLATION  --  which property breaks when each element is removed")
print("=" * 78)

ABL_C = 0.5
ABL_STEPS = steps(150)
abl_reference = evaluate_loop_pair(RIDGE_TEST)
abl_instance = RIDGE_TEST[0]

# Trained variants, one per architectural switch setting.
abl_variants = [
    ('PNS (full, trained)',   dict(use_gate=True,  use_frame=True, use_h2=True,  bounded=True)),
    ('no h^2 scaling',        dict(use_gate=True,  use_frame=True, use_h2=False, bounded=True)),
    ('no curvature gate',     dict(use_gate=False, use_frame=True, use_h2=True,  bounded=True)),
    ('unbounded output',      dict(use_gate=True,  use_frame=True, use_h2=True,  bounded=False)),
]

abl_nets = {}
for label, sw in abl_variants:
    tag = 'ablation_' + label.replace(' ', '_').replace('^', '').replace('(', '').replace(')', '').replace(',', '')
    abl_nets[label], _ = cached_model(
        tag,
        build_fn=(lambda sw=sw: _build_ablation_net(sw, ABL_C)),
        train_fn=(lambda sw=sw: train_operator(
            RIDGE_TRAIN, RIDGE_TEST, C=ABL_C, weights=RIDGE_WEIGHTS,
            n_steps=ABL_STEPS, lr=2e-3, seed=7, verbose=False,
            bounded=sw['bounded'], use_gate=sw['use_gate'],
            use_frame=sw['use_frame'], use_h2=sw['use_h2'])),
        meta=dict(switches=sw, C=ABL_C, n_steps=ABL_STEPS))

# Untrained full architecture, to separate architecture from training.
torch.manual_seed(123)
net_random = CorrectionNet(dim_in=5, hidden=32, n_layers=2, C=ABL_C)

abl_rows = []
for label, sw in abl_variants:
    net = abl_nets[label]
    rows = ablation_one_step(RIDGE_TEST, net, ABL_C,
                             use_gate=sw['use_gate'], use_frame=sw['use_frame'],
                             use_h2=sw['use_h2'])
    prox = max(r['prox_ratio_max'] for r in rows)
    affine = certify_affine_reproduction_switched(net, sw)
    near = paired_summary(abl_reference, rows, 'sdf_rmse_near')
    op = make_subdivider(net, use_gate=sw['use_gate'], use_frame=sw['use_frame'],
                         use_h2=sw['use_h2'])
    rep_ratio, rep_jump = repeated_summary(abl_instance, op, n_levels=3, C_arch=ABL_C)
    abl_rows.append(dict(variant=label,
                         max_prox_ratio_one_step=f'{prox:.4f}',
                         affine_reproduction_error=f'{affine:.3e}',
                         max_prox_ratio_level3=f'{rep_ratio:.4f}',
                         max_normal_jump_level3=f'{rep_jump:.4f}',
                         sdf_near_improvement_pct=f"{near['rel_improvement_pct']:+.2f}"))

# Random-weight row: same architecture, no training.
rows = ablation_one_step(RIDGE_TEST, net_random, ABL_C)
near = paired_summary(abl_reference, rows, 'sdf_rmse_near')
rep_ratio, rep_jump = repeated_summary(abl_instance, make_subdivider(net_random),
                                       n_levels=3, C_arch=ABL_C)
abl_rows.insert(1, dict(variant='PNS with random weights',
                        max_prox_ratio_one_step=f"{max(r['prox_ratio_max'] for r in rows):.4f}",
                        affine_reproduction_error=f'{certify_affine_reproduction(net_random):.3e}',
                        max_prox_ratio_level3=f'{rep_ratio:.4f}',
                        max_normal_jump_level3=f'{rep_jump:.4f}',
                        sdf_near_improvement_pct=f"{near['rel_improvement_pct']:+.2f}"))

# Unconstrained multilayer perceptron: every architectural element removed at once.
net_abl_unc, _ = cached_model(
    'ablation_unconstrained',
    build_fn=lambda: UnconstrainedNet(hidden=32, n_layers=3),
    train_fn=lambda: train_unconstrained(RIDGE_TRAIN, n_steps=ABL_STEPS, lr=2e-3, seed=7),
    meta=dict(n_steps=ABL_STEPS))
rows = evaluate_unconstrained_pair(RIDGE_TEST, net_abl_unc, C_for_ratio=ABL_C)
near = paired_summary(abl_reference, rows, 'sdf_rmse_near')
with torch.no_grad():
    q_pl, q0_pl = net_abl_unc(planar_patch())
    unc_affine = float((q_pl - q0_pl).norm(dim=1).max())
rep_ratio, rep_jump = repeated_summary(
    abl_instance, lambda m: subdivide_unconstrained(m, net_abl_unc), n_levels=3, C_arch=ABL_C)
abl_rows.append(dict(variant='unconstrained multilayer perceptron',
                     max_prox_ratio_one_step=f"{max(r['prox_ratio_max'] for r in rows):.4f}",
                     affine_reproduction_error=f'{unc_affine:.3e}',
                     max_prox_ratio_level3=f'{rep_ratio:.4f}',
                     max_normal_jump_level3=f'{rep_jump:.4f}',
                     sdf_near_improvement_pct=f"{near['rel_improvement_pct']:+.2f}"))

print()
hdr = (f"{'Variant':<38s} {'prox L1':>9s} {'affine err':>12s} {'prox L3':>9s} "
       f"{'jump L3':>9s} {'SDF near':>10s}")
print(hdr)
print('-' * len(hdr))
for r in abl_rows:
    print(f"{r['variant']:<38s} {r['max_prox_ratio_one_step']:>9s} "
          f"{r['affine_reproduction_error']:>12s} {r['max_prox_ratio_level3']:>9s} "
          f"{r['max_normal_jump_level3']:>9s} {r['sdf_near_improvement_pct']+'%':>10s}")

write_table('ablation',
            ['variant', 'max_prox_ratio_one_step', 'affine_reproduction_error',
             'max_prox_ratio_level3', 'max_normal_jump_level3',
             'sdf_near_improvement_pct'], abl_rows)

print("\n  Each architectural element protects a specific property. Removing the h^2")
print("  scaling breaks asymptotic proximity, removing the gate breaks affine")
print("  reproduction, removing the bound breaks the envelope, and removing the whole")
print("  construction removes every guarantee while improving the one-step fit.")

RESULTS['ablation'] = abl_rows

## 9. Headline experimentA single member of the constrained operator class is trained on twenty randomridges at $C = 0.5$. The unconstrained foil is trained on the same data with thesame step count. Both are evaluated against Loop on the ten held-out instanceswith paired bootstrap confidence intervals.The expected pattern is that the unconstrained foil produces larger one-stepgeometric improvements, because free vertex prediction has more capacity to fita target than gated bounded correction, while degrading the global normal erroras a signature of high-frequency artefacts. PNS improves signed-distance andHausdorff error while leaving normal error essentially unchanged, and staysinside its envelope throughout.

In [ ]:
print("=" * 72)
print("HEADLINE TRAINING -- Gaussian ridges, C = 0.5, ridge-tuned weights")
print("=" * 72)
net_main, hist_main = cached_model(
    'pns_headline_C0.5',
    build_fn=lambda: CorrectionNet(dim_in=5, hidden=32, n_layers=2, C=0.5),
    train_fn=lambda: train_operator(
        RIDGE_TRAIN, RIDGE_TEST, C=0.5, weights=RIDGE_WEIGHTS,
        n_steps=steps(250), lr=2e-3, seed=1, log_every=50),
    meta=dict(C=0.5, hidden=32, n_layers=2, n_steps=steps(250), seed=1))

print()
print("UNCONSTRAINED BASELINE -- raw-displacement MLP, 250 steps")
net_unc, hist_unc = cached_model(
    'unconstrained_foil',
    build_fn=lambda: UnconstrainedNet(hidden=32, n_layers=3),
    train_fn=lambda: train_unconstrained(RIDGE_TRAIN, n_steps=steps(250), lr=2e-3, seed=1),
    meta=dict(hidden=32, n_layers=3, n_steps=steps(250), seed=1))

# Evaluate
loop_rows  = evaluate_loop_pair(RIDGE_TEST)
main_rows  = evaluate_constrained_pair(RIDGE_TEST, net_main, C=0.5)
unc_rows   = evaluate_unconstrained_pair(RIDGE_TEST, net_unc, C_for_ratio=0.5)


# Pretty-print results table
print()
print("=" * 80)
print("RIDGE TEST SET (10 held-out instances)  --  paired statistics")
print("=" * 80)
def print_metric(name, rows_loop, rows_other, label):
    s = paired_summary(rows_loop, rows_other, name)
    sign = '+' if s['rel_improvement_pct'] >= 0 else ''
    print(f"  {label:<12s}  Loop={s['mean_a']:.5f}  trained={s['mean_b']:.5f}  "
          f"improvement = {sign}{s['rel_improvement_pct']:.2f}%  "
          f"(95% CI [{s['ci_lo_pct']:+.2f}%, {s['ci_hi_pct']:+.2f}%])")

print()
print("Trained S_theta vs Loop:")
for k, lbl in [('sdf_rmse_near', 'SDF near'), ('sdf_rmse', 'SDF global'),
               ('hausdorff_near','Haus near'), ('hausdorff', 'Haus global'),
               ('normal_err_near','Normal near'), ('normal_err', 'Normal global')]:
    print_metric(k, loop_rows, main_rows, lbl)

# Structural metrics
prox_main = np.array([r['prox_ratio_max'] for r in main_rows])
prox_unc  = np.array([r['prox_ratio_max'] for r in unc_rows])
print()
print(f"  Prox ratio (max over test set):   trained={prox_main.max():.4f} <= 1 OK  "
      f"unconstrained={prox_unc.max():.4f} (no architectural cap)")
print(f"  Prox ratio (mean over test set):  trained={prox_main.mean():.4f}  "
      f"unconstrained={prox_unc.mean():.4f}")

# Summary block: also show unconstrained vs Loop for context
print()
print("Unconstrained neural foil vs Loop:")
for k, lbl in [('sdf_rmse_near', 'SDF near'), ('sdf_rmse', 'SDF global'),
               ('normal_err_near','Normal near')]:
    print_metric(k, loop_rows, unc_rows, lbl)

# Save results
RESULTS['headline']['loop_rows']  = loop_rows
RESULTS['headline']['main_rows']  = main_rows
RESULTS['headline']['unc_rows']   = unc_rows
RESULTS['headline']['paired'] = {
    k: paired_summary(loop_rows, main_rows, k)
    for k in ['sdf_rmse', 'sdf_rmse_near', 'hausdorff', 'hausdorff_near',
              'normal_err', 'normal_err_near']
}



# ---- Held-out ridge comparison table ----
headline_rows = []
for key, label in [('sdf_rmse_near', 'SDF RMSE near feature'),
                   ('sdf_rmse', 'SDF RMSE global'),
                   ('hausdorff_near', 'Hausdorff near feature'),
                   ('hausdorff', 'Hausdorff global'),
                   ('normal_err_near', 'Normal error near feature'),
                   ('normal_err', 'Normal error global')]:
    s_pns = paired_summary(loop_rows, main_rows, key)
    s_unc = paired_summary(loop_rows, unc_rows, key)
    headline_rows.append(dict(
        metric=label,
        loop_mean=f"{s_pns['mean_a']:.6f}",
        pns_mean=f"{s_pns['mean_b']:.6f}",
        pns_improvement_pct=f"{s_pns['rel_improvement_pct']:+.2f}",
        pns_ci=f"[{s_pns['ci_lo_pct']:+.2f}, {s_pns['ci_hi_pct']:+.2f}]",
        unconstrained_mean=f"{s_unc['mean_b']:.6f}",
        unconstrained_improvement_pct=f"{s_unc['rel_improvement_pct']:+.2f}",
        unconstrained_ci=f"[{s_unc['ci_lo_pct']:+.2f}, {s_unc['ci_hi_pct']:+.2f}]"))

write_table('main_results_table',
            ['metric', 'loop_mean', 'pns_mean', 'pns_improvement_pct', 'pns_ci',
             'unconstrained_mean', 'unconstrained_improvement_pct', 'unconstrained_ci'],
            headline_rows)

## 10. Headline figurePaired relative improvements over Loop on the held-out test set, separated intonear-feature and global regions, with ninety-five percent confidence intervals.

In [ ]:
# ----- Figure: paired improvements with CI -----
fig, ax = plt.subplots(figsize=(11.0, 5.4), constrained_layout=True)

metric_groups = [
    ('SDF RMSE\n(near feature)',  'sdf_rmse_near'),
    ('SDF RMSE\n(global)',        'sdf_rmse'),
    ('Hausdorff\n(near feature)', 'hausdorff_near'),
    ('Hausdorff\n(global)',       'hausdorff'),
    ('Normal err\n(near feature)','normal_err_near'),
    ('Normal err\n(global)',      'normal_err'),
]

x = np.arange(len(metric_groups))
w = 0.36
main_means, main_los, main_his = [], [], []
unc_means,  unc_los,  unc_his  = [], [], []
for _, key in metric_groups:
    sm = paired_summary(loop_rows, main_rows, key)
    su = paired_summary(loop_rows, unc_rows,  key)
    main_means.append(sm['rel_improvement_pct'])
    main_los.append(sm['rel_improvement_pct'] - sm['ci_lo_pct'])
    main_his.append(sm['ci_hi_pct'] - sm['rel_improvement_pct'])
    unc_means.append(su['rel_improvement_pct'])
    unc_los.append(su['rel_improvement_pct'] - su['ci_lo_pct'])
    unc_his.append(su['ci_hi_pct'] - su['rel_improvement_pct'])

ax.bar(x - w/2, main_means, w, yerr=[main_los, main_his],
       color=PALETTE[2], edgecolor='white', linewidth=1, capsize=4,
       label=r'Trained $S_\theta$ (C=0.5)')
ax.bar(x + w/2, unc_means, w, yerr=[unc_los, unc_his],
       color=PALETTE[3], edgecolor='white', linewidth=1, capsize=4,
       label='Unconstrained neural foil')
ax.axhline(0, color='k', lw=0.6)

ax.set_xticks(x)
ax.set_xticklabels([g[0] for g in metric_groups], fontsize=9)
ax.set_ylabel('Relative improvement over Loop (%)')
ax.set_title('Headline: paired improvement on 10 held-out ridge instances (95% CI)')
ax.grid(True, axis='y', alpha=0.25)

# Annotate trained S_theta values above bars
for i, (m, lo, hi) in enumerate(zip(main_means, main_los, main_his)):
    ypos = m + hi + 1.5 if m >= 0 else m - lo - 3
    ax.text(x[i] - w/2, ypos, f'{m:+.1f}%', ha='center', fontsize=8.5,
            color=PALETTE[2], fontweight='bold')

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20),
          ncol=2, framealpha=0.95, frameon=True,
          handlelength=2.0, columnspacing=1.6)

fig.savefig(OUT / 'headline_ridge_improvements.png', dpi=180, bbox_inches='tight')
plt.show()

## 11. Cross-section figureOne representative held-out instance with a sharp ridge. The left panel showsthe cross-section along the full patch, the middle panel zooms on thenear-feature region, and the right panel shows the per-vertex signed-distanceerror. Loop underfits the ridge cap because the inserted edge vertex receivesonly one-eighth weight from the diagonal cross of opposite vertices. Trained PNSlifts the cap without overshooting, while the unconstrained foil tracks the peakmost closely and retains visible residual error at the shoulders.

In [ ]:
# Pick a representative test mesh: one with high curvature (large b) for visual clarity
def select_money_instance(samples, prefer_high_b=True):
    if prefer_high_b:
        return max(range(len(samples)), key=lambda i: samples[i]['params']['b'])
    return 0

mi = select_money_instance(RIDGE_TEST)
sample = RIDGE_TEST[mi]
m = sample['mesh']; sdf_fn = sample['sdf']; pp = sample['params']
print(f"Selected instance {mi}: a={pp['a']:.3f} b={pp['b']:.2f} phi={pp['phi']:.3f}")

# Compute insertion points for each method
edges, edge_faces, opps = build_edge_data(m.F)
q_loop = loop_edge_vertices(m.V, edges, opps)
with torch.no_grad():
    q_main, _ = constrained_edge_vertices(m, net_main, c_gate=C_GATE)
    q_unc, _  = net_unc(m)

# Project to (y', z) via the rotation
cos_p, sin_p = math.cos(pp['phi']), math.sin(pp['phi'])
def to_yp_z(P):
    x = P[:, 0] - pp['cx']; y = P[:, 1] - pp['cy']
    return (-sin_p * x + cos_p * y).numpy(), P[:, 2].numpy()


# Mesh vertices for context (faintly drawn)
yp_mesh, z_mesh = to_yp_z(m.V)

# Inserted vertices
yp_loop, z_loop = to_yp_z(q_loop)
yp_main, z_main = to_yp_z(q_main)
yp_unc,  z_unc  = to_yp_z(q_unc)

# Analytic ridge curve in (y', z)
yp_grid = np.linspace(-1.2, 1.2, 400)
z_curve = pp['a'] * np.exp(-pp['b'] * yp_grid ** 2)

# Per-vertex SDF errors at each method's inserted vertices (for stem panel)
err_loop = sdf_fn(q_loop).abs().numpy()
err_main = sdf_fn(q_main).abs().numpy()
err_unc  = sdf_fn(q_unc).abs().numpy()


# Three-panel: full view + near-feature zoom + per-vertex SDF error
# Three-panel: full view + near-feature zoom + per-vertex SDF error
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 3, figsize=(17.5, 5.0),
                         gridspec_kw={'width_ratios': [1, 1, 1.05]},
                         constrained_layout=True)

for axi, (xrange, title) in enumerate([
    ((-1.05, 1.05), 'Cross-section (full patch)'),
    ((-0.5, 0.5),   'Cross-section (near feature, zoomed)'),
]):
    ax = axes[axi]
    ax.plot(yp_grid, z_curve, '-', color='black', lw=1.6, zorder=4)
    ax.scatter(yp_mesh, z_mesh, color='0.7', s=8, alpha=0.4, zorder=1)
    ax.scatter(yp_loop, z_loop, color=PALETTE[0], s=22, marker='o',
               zorder=3, alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.scatter(yp_main, z_main, color=PALETTE[2], s=22, marker='s',
               zorder=3, alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.scatter(yp_unc, z_unc, color=PALETTE[3], s=22, marker='^',
               zorder=3, alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.set_xlim(*xrange)
    ax.set_xlabel(r"$y'$ (across-ridge coordinate)")
    if axi == 0:
        ax.set_ylabel(r'$z$ (height)')
    ax.set_title(title)
    ax.grid(True, alpha=0.25)

# Third panel: per-vertex SDF error magnitude as stem plot
ax = axes[2]
ax.axvspan(-0.3, 0.3, color='0.92', alpha=0.5, zorder=0)
ax.vlines(yp_loop, 0, err_loop, color=PALETTE[0], lw=0.5, alpha=0.5)
ax.scatter(yp_loop, err_loop, color=PALETTE[0], marker='o', s=22,
           edgecolor='white', linewidth=0.5, zorder=3)
ax.vlines(yp_main, 0, err_main, color=PALETTE[2], lw=0.5, alpha=0.5)
ax.scatter(yp_main, err_main, color=PALETTE[2], marker='s', s=22,
           edgecolor='white', linewidth=0.5, zorder=3)
ax.vlines(yp_unc, 0, err_unc, color=PALETTE[3], lw=0.5, alpha=0.5)
ax.scatter(yp_unc, err_unc, color=PALETTE[3], marker='^', s=22,
           edgecolor='white', linewidth=0.5, zorder=3)
ax.set_xlim(-1.05, 1.05)
ax.set_ylim(bottom=0)
ax.set_xlabel(r"$y'$ (across-ridge coordinate)")
ax.set_ylabel(r'$|d_\Sigma(q)|$  per inserted vertex')
ax.set_title('Per-vertex SDF error')
ax.grid(True, alpha=0.25)

# Shared legend below all three panels.
legend_handles = [
    Line2D([0], [0], color='black', lw=1.6, label='analytic ridge'),
    Line2D([0], [0], marker='.', color='0.7', linestyle='None',
           markersize=7, label='mesh vertices'),
    Line2D([0], [0], marker='o', color=PALETTE[0], linestyle='None',
           markersize=7, markeredgecolor='white', markeredgewidth=0.5,
           label='Loop'),
    Line2D([0], [0], marker='s', color=PALETTE[2], linestyle='None',
           markersize=7, markeredgecolor='white', markeredgewidth=0.5,
           label=r'Trained $S_\theta$'),
    Line2D([0], [0], marker='^', color=PALETTE[3], linestyle='None',
           markersize=7, markeredgecolor='white', markeredgewidth=0.5,
           label='Unconstrained neural foil'),
    Patch(facecolor='0.92', edgecolor='0.6', alpha=0.7,
          label='near-feature region (right panel)'),
]
fig.legend(handles=legend_handles,
           loc='outside lower center',
           ncol=6, frameon=False,
           handlelength=2.0, columnspacing=1.8, fontsize=10)

# Quick numerical summary printed below the figure
print(f"  median |SDF error| in near-feature region (|y'|<0.3):")
near_loop = err_loop[np.abs(yp_loop) < 0.3]
near_main = err_main[np.abs(yp_main) < 0.3]
near_unc  = err_unc[np.abs(yp_unc) < 0.3]
print(f"    Loop          : {np.median(near_loop):.5f}")
print(f"    Trained S_t   : {np.median(near_main):.5f}  ({(np.median(near_loop)-np.median(near_main))/np.median(near_loop)*100:+.1f}% vs Loop)")
print(f"    Unconstrained : {np.median(near_unc):.5f}  ({(np.median(near_loop)-np.median(near_unc))/np.median(near_loop)*100:+.1f}% vs Loop)")

fig.suptitle(f'Money figure -- instance {mi}: a={pp["a"]:.2f}, b={pp["b"]:.1f}')
fig.savefig(OUT / 'money_cross_section.png', dpi=180, bbox_inches='tight')
plt.show()

## 12. Budget localisationEach edge is coloured by its proximity ratio $\|q_i^\theta - q_i^0\| / (C h_i^2)$.Edges away from the ridge should be near zero because the gate is dormant, andedges near the ridge should approach saturation.

In [ ]:
# Use the same money instance
# Use the same money instance
fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.6), constrained_layout=True)

# Compute per-edge proximity ratios for trained S_theta
with torch.no_grad():
    q_t, q_0, aux = constrained_edge_vertices(m, net_main, c_gate=C_GATE, return_aux=True)
edges = aux['edges']; e_len = aux['e_len']
h2 = (e_len ** 2).clamp_min(1e-12)
ratio_per_edge = ((q_t - q_0).norm(dim=1) / (net_main.C * h2)).numpy()
gate_per_edge  = aux['gate'].numpy()

# Edge midpoints in xy for plotting (using the original mesh vertices)
pa = m.V[edges[:, 0]]; pb = m.V[edges[:, 1]]
mid_xy = ((pa + pb) / 2).numpy()

# Compute y' for each edge (for sorting / annotation)
def yprime_xy(p_xy):
    x = p_xy[:, 0] - pp['cx']; y = p_xy[:, 1] - pp['cy']
    return -sin_p * x + cos_p * y
yp_edges = yprime_xy(mid_xy)

# Left: budget heatmap (xy view, edges colored by prox ratio)
ax = axes[0]
from matplotlib.collections import LineCollection
segs = np.stack([pa.numpy()[:, :2], pb.numpy()[:, :2]], axis=1)
norm = plt.Normalize(vmin=0, vmax=max(ratio_per_edge.max(), 1.0))
lc = LineCollection(segs, cmap='viridis', norm=norm, linewidths=2.4)
lc.set_array(ratio_per_edge)
ax.add_collection(lc)
ax.scatter(m.V[:, 0], m.V[:, 1], color='0.4', s=8, alpha=0.7, zorder=2)
along = np.linspace(-1.2, 1.2, 100)
ridge_x = pp['cx'] + along * math.cos(pp['phi'])
ridge_y = pp['cy'] + along * math.sin(pp['phi'])
ax.plot(ridge_x, ridge_y, 'r--', lw=1.2, alpha=0.7, zorder=3, label='ridge centerline')

ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05)
ax.set_aspect('equal')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Per-edge proximity-budget usage')
ax.legend(loc='upper left', framealpha=0.95)
cbar = plt.colorbar(lc, ax=ax, fraction=0.045, pad=0.02)
cbar.set_label(r'$\|q_i^\theta - q_i^0\| / (C h_i^2)$')

# Right: prox ratio vs |y'|
ax = axes[1]
ax.scatter(np.abs(yp_edges), ratio_per_edge, s=24, color=PALETTE[2],
           alpha=0.7, edgecolor='white', linewidth=0.5)
ax.axhline(1.0, color='k', ls='--', alpha=0.6, lw=1.0, label='cap (= 1)')
ax.axvline(0.3, color='gray', ls=':', alpha=0.6, lw=1.0,
           label=r"near-feature ($|y'| = 0.3$)")
ax.set_xlabel(r"distance from ridge centerline $|y'|$")
ax.set_ylabel(r'$\|q_i^\theta - q_i^0\| / (C h_i^2)$')
ax.set_title('Prox ratio concentrates near the feature')
ax.set_xlim(-0.02, 1.05)
ax.set_ylim(0, max(1.05, ratio_per_edge.max() * 1.1))
ax.grid(True, alpha=0.25)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.16),
          ncol=2, framealpha=0.95, frameon=True,
          handlelength=2.0, columnspacing=1.6, fontsize=9)

fig.savefig(OUT / 'budget_utilisation.png', dpi=180, bbox_inches='tight')
plt.show()

# Print a summary
near_mask = (np.abs(yp_edges) < 0.3)
print(f"Near-feature edges (|y'|<0.3):  prox ratio mean = {ratio_per_edge[near_mask].mean():.4f},  max = {ratio_per_edge[near_mask].max():.4f}")

# Print a summary
near_mask = (np.abs(yp_edges) < 0.3)
print(f"Near-feature edges (|y'|<0.3):  prox ratio mean = {ratio_per_edge[near_mask].mean():.4f},  max = {ratio_per_edge[near_mask].max():.4f}")
print(f"Far-from-feature edges       :  prox ratio mean = {ratio_per_edge[~near_mask].mean():.4f},  max = {ratio_per_edge[~near_mask].max():.4f}")
print(f"Gate value mean (near vs far): {gate_per_edge[near_mask].mean():.3f} vs {gate_per_edge[~near_mask].mean():.3f}")

### 12.1 Budget verificationTwo clusters carry non-trivial proximity ratio, one at the ridge crest and oneat the shoulder where the second derivative changes sign. The panels below testwhether both are gate-driven rather than artefacts, by plotting the proximityratio directly against the gate value and by colouring each edge by itsalignment with the ridge tangent.

In [ ]:
# ---- Verification: prox ratio vs gate value, with edge-orientation overlay ----
edges_int = aux['edges']
pa_idx = edges_int[:, 0].numpy(); pb_idx = edges_int[:, 1].numpy()
pa_xy = m.V[pa_idx, :2].numpy(); pb_xy = m.V[pb_idx, :2].numpy()
edge_vec = pb_xy - pa_xy
edge_dir = edge_vec / (np.linalg.norm(edge_vec, axis=1, keepdims=True) + 1e-10)
ridge_dir = np.array([math.cos(pp['phi']), math.sin(pp['phi'])])
edge_alignment = np.abs(edge_dir @ ridge_dir)   # 1 = parallel to ridge, 0 = perpendicular

# y' of each endpoint (and span)
yp_a = -sin_p * (pa_xy[:, 0] - pp['cx']) + cos_p * (pa_xy[:, 1] - pp['cy'])
yp_b = -sin_p * (pb_xy[:, 0] - pp['cx']) + cos_p * (pb_xy[:, 1] - pp['cy'])
yp_span = np.abs(yp_a - yp_b)
yp_mid = 0.5 * (yp_a + yp_b)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4), constrained_layout=True)

# Left: prox ratio vs gate, colored by edge alignment
sc = axes[0].scatter(gate_per_edge, ratio_per_edge, s=26, c=edge_alignment,
                      cmap='coolwarm', alpha=0.8, edgecolor='white', linewidth=0.4,
                      vmin=0, vmax=1)
axes[0].plot([0, 1], [0, 1], 'k--', lw=0.8, alpha=0.5, label=r'$y = x$')
axes[0].set_xlabel(r'Gate value $\gamma_i = \tanh(c\,\|\varphi^{\mathrm{curv}}_i\|^2)$')
axes[0].set_ylabel(r'Proximity ratio $\|q_i^\theta - q_i^0\| / (C h_i^2)$')
axes[0].set_title('Prox ratio = gate -- network saturates the bound')
axes[0].legend(loc='upper left', framealpha=0.95)
axes[0].grid(True, alpha=0.25)
cbar = plt.colorbar(sc, ax=axes[0])
cbar.set_label(r'$|$edge dir $\cdot$ ridge dir$|$  (1=parallel, 0=perp.)')

# Right: prox ratio vs |y'_mid|, same colour code
sc = axes[1].scatter(np.abs(yp_mid), ratio_per_edge, s=26, c=edge_alignment,
                      cmap='coolwarm', alpha=0.8, edgecolor='white', linewidth=0.4,
                      vmin=0, vmax=1)
axes[1].axhline(1.0, color='k', ls='--', alpha=0.6, lw=1.0,
                label='architectural cap (= 1)')
axes[1].axvline(0.3, color='gray', ls=':', alpha=0.6, lw=1.0,
                label="near-feature boundary")
axes[1].set_xlabel(r"$|y'_{\rm mid}|$  (distance of edge midpoint from ridge centerline)")
axes[1].set_ylabel(r'Proximity ratio')
axes[1].set_title('Both clusters: parallel-to-ridge edges at curvature peaks')
axes[1].set_ylim(0, 1.1)
axes[1].grid(True, alpha=0.25)
plt.colorbar(sc, ax=axes[1])
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.16),
               ncol=2, framealpha=0.95, frameon=True,
               handlelength=2.0, columnspacing=1.6, fontsize=9)

fig.savefig(OUT / 'budget_verification.png', dpi=180, bbox_inches='tight')
plt.show()

# Statistical confirmation
first_cluster  = (np.abs(yp_mid) < 0.15) & (ratio_per_edge > 0.5)
second_cluster = (np.abs(yp_mid) > 0.35) & (ratio_per_edge > 0.1)
quiet_far      = (np.abs(yp_mid) > 0.7)

# Statistical confirmation
first_cluster  = (np.abs(yp_mid) < 0.15) & (ratio_per_edge > 0.5)
second_cluster = (np.abs(yp_mid) > 0.35) & (ratio_per_edge > 0.1)
quiet_far      = (np.abs(yp_mid) > 0.7)

# Pearson correlation between gate and prox ratio
corr = np.corrcoef(gate_per_edge, ratio_per_edge)[0, 1]
print(f"  Pearson(gate, prox ratio) = {corr:.4f}  (1.0 means network has fully saturated the bound)")
print()
print(f"  First cluster  (|y'|<0.15, ratio>0.5):  {int(first_cluster.sum()):3d} edges,  "
      f"mean align = {edge_alignment[first_cluster].mean():.2f},  "
      f"mean y' span = {yp_span[first_cluster].mean():.3f}")
print(f"  Second cluster (|y'|>0.35, ratio>0.1): {int(second_cluster.sum()):3d} edges,  "
      f"mean align = {edge_alignment[second_cluster].mean():.2f},  "
      f"mean y' span = {yp_span[second_cluster].mean():.3f}")
print(f"  Quiet far      (|y'|>0.7):              {int(quiet_far.sum()):3d} edges,  "
      f"mean align = {edge_alignment[quiet_far].mean():.2f},  "
      f"mean y' span = {yp_span[quiet_far].mean():.3f}")
print()
print("  Interpretation:")
print("  - Pearson = 1.0: prox ratio is exactly the gate value (eta has saturated to C).")
print("    The bounded head is binding; no other free parameters affect displacement magnitude.")
print("  - Both high-ratio clusters consist of edges PARALLEL to the ridge axis (align ~ 1),")
print("    sitting at fixed y' values with negligible y' span. These edges sample LOCAL")
print("    perpendicular curvature without averaging across feature regions.")
print("  - Cluster 1 (y'~0): parallel edges on the ridge crest, where the perpendicular second")
print("    derivative -2ab*exp(-b*y'^2) is largest (in absolute value). Gate saturates.")
print("  - Cluster 2 (y'~0.5): parallel edges past the inflection (y'_inflect ~ 1/sqrt(2b)),")
print("    where the perpendicular second derivative is moderate but nonzero. Gate fires partially.")
print("  - Quiet far (y'>0.7): same orientation, but the surface is essentially flat. Gate ~ 0.")
print("  Conclusion: the gate detects local perpendicular surface curvature exactly as designed,")
print("  regardless of edge orientation. The second cluster is a genuine architectural response")
print("  to a real curvature feature, not an artifact.")

## 13. Repeated subdivisionA subdivision scheme is defined by its behaviour under repeated application, soeach operator is applied four times to the held-out ridges. Three quantities aretracked at every level. The signed-distance error measures how close the mesh isto the analytic surface. The maximum proximity ratio measures how much of thearchitectural envelope is used, relative to the parent level's edge lengths. Themaximum face-normal jump measures mesh regularity, with values approaching $\pi$indicating flipped faces.This is the central empirical result. The bound is enforced per level againstthe parent's edge lengths, so a scheme that stays inside the envelope at everylevel remains Loop-proximate no matter how many times it is applied.

In [ ]:
@torch.no_grad()
def evaluate_at_level(mesh, sdf_fn, params, C_arch=0.5):
    """Evaluate metrics on the *current* mesh (no subdivision in this call)."""
    edges, edge_faces, _ = build_edge_data(mesh.F)
    _, e_len = edge_frames(mesh.V, mesh.F, edges, edge_faces)
    h_max = float(e_len.max())
    d = sdf_fn(mesh.V)
    rmse = float(torch.sqrt((d ** 2).mean()))
    Fn = _face_normals(mesh.V, mesh.F)
    Fn_left  = Fn[edge_faces[:, 0]]
    Fn_right = Fn[edge_faces[:, 1]]
    cos_dihedral = (Fn_left * Fn_right).sum(dim=1).clamp(-1, 1)
    max_normal_jump = float(torch.acos(cos_dihedral).max())
    return dict(h_max=h_max, sdf_rmse=rmse, max_normal_jump=max_normal_jump)


def repeated_subdivision_test(samples, op_fn, op_name, n_levels=4, C_arch=0.5):
    """Apply op_fn n_levels times; record metrics at each level.

    Per-level proximity ratio is measured against the parent's classical Loop
    reference (all edges, including boundary), against the architectural
    target C_arch * h^2 of the parent's edge lengths."""
    results = defaultdict(list)
    for s in samples:
        m = s['mesh']
        per_level = [evaluate_at_level(m, s['sdf'], s.get('params'), C_arch=C_arch)]
        prox_ratios_per_level = []
        for L in range(n_levels):
            parent = m
            edge_keys, _, edge_opps = _build_full_edge_index(parent.F)
            q_0_full = _new_edge_vertices_loop(parent.V, edge_keys, edge_opps)
            e_len_full = torch.tensor(
                [float(torch.norm(parent.V[a] - parent.V[b])) for a, b in edge_keys],
                dtype=DTYPE)
            h2 = (e_len_full ** 2).clamp_min(1e-12)
            m = op_fn(parent)
            n_old = parent.n_v
            q_t_full = m.V[n_old:]
            ratio_max = float(((q_t_full - q_0_full).norm(dim=1) / (C_arch * h2)).max())
            prox_ratios_per_level.append(ratio_max)
            per_level.append(evaluate_at_level(m, s['sdf'], s.get('params'), C_arch=C_arch))
        for L in range(n_levels + 1):
            results['h_max'].append(per_level[L]['h_max'])
            results['sdf_rmse'].append(per_level[L]['sdf_rmse'])
            results['max_normal_jump'].append(per_level[L]['max_normal_jump'])
            results['level'].append(L)
            results['op'].append(op_name)
            results['instance'].append(s['name'])
            results['prox_ratio_max'].append(prox_ratios_per_level[L-1] if L >= 1 else 0.0)
    return results


# Run the test on the ridge test set
print("=" * 70)
print("REPEATED-SUBDIVISION STABILITY (4 levels on 10 held-out ridges)")
print("=" * 70)
t0 = time.time()
res_loop = repeated_subdivision_test(RIDGE_TEST, subdivide_loop, 'Loop', n_levels=4, C_arch=0.5)
res_main = repeated_subdivision_test(RIDGE_TEST,
    lambda m: subdivide_constrained(m, net_main), 'Constrained', n_levels=4, C_arch=0.5)
res_unc  = repeated_subdivision_test(RIDGE_TEST,
    lambda m: subdivide_unconstrained(m, net_unc), 'Unconstrained', n_levels=4, C_arch=0.5)
print(f"  done in {time.time()-t0:.1f}s")


# Aggregate per-level means
def aggregate_by_level(res):
    by_level = defaultdict(lambda: defaultdict(list))
    for i, lvl in enumerate(res['level']):
        for k in ['h_max', 'sdf_rmse', 'max_normal_jump', 'prox_ratio_max']:
            by_level[lvl][k].append(res[k][i])
    levels = sorted(by_level.keys())
    out = {}
    for lvl in levels:
        out[lvl] = {k: float(np.mean(by_level[lvl][k])) for k in ['h_max', 'sdf_rmse', 'max_normal_jump', 'prox_ratio_max']}
    return out

agg_loop = aggregate_by_level(res_loop)
agg_main = aggregate_by_level(res_main)
agg_unc  = aggregate_by_level(res_unc)

# Print summary
print()
print(f"{'Level':<8s}  {'Op':<14s}  {'h_max':>8s}  {'SDF RMSE':>10s}  {'Max prox ratio':>14s}  {'Max norm jump':>13s}")
print('-' * 78)
for op_name, agg in [('Loop', agg_loop), ('Constrained', agg_main), ('Unconstrained', agg_unc)]:
    for lvl in sorted(agg.keys()):
        a = agg[lvl]
        prox = a['prox_ratio_max'] if lvl >= 1 else float('nan')
        prox_s = f"{prox:14.4f}" if not math.isnan(prox) else f"{'--':>14s}"
        print(f"{lvl:<8d}  {op_name:<14s}  {a['h_max']:8.4f}  {a['sdf_rmse']:10.5f}  "
              f"{prox_s}  {a['max_normal_jump']:13.4f}")
    print()

# ----- Repeated-subdivision figure -----
fig, axes = plt.subplots(1, 3, figsize=(15.5, 5.4), constrained_layout=True)

levels = sorted(agg_loop.keys())
last_handles, last_labels = [], []
for ax, key, ylabel, title, log in zip(
    axes, ['sdf_rmse', 'prox_ratio_max', 'max_normal_jump'],
    ['SDF RMSE', r'$\max_i \|q_i^\theta - q_i^0\| / (C h_i^2)$', 'Max face-normal jump (rad)'],
    ['Approximation error', 'Proximity ratio at each level', 'Mesh regularity'],
    [True, True, False]):
    for op_name, agg, color, marker in [
        ('Loop',           agg_loop, PALETTE[0], 'o'),
        (r'Trained $S_\theta$', agg_main, PALETTE[2], 's'),
        ('Unconstrained',  agg_unc,  PALETTE[3], '^'),
    ]:
        ys = [agg[L][key] for L in levels]
        if key == 'prox_ratio_max':
            ys = [agg[L][key] if L >= 1 else float('nan') for L in levels]
        plot_fn = ax.semilogy if log else ax.plot
        plot_fn(levels, ys, marker + '-', color=color, lw=1.6, markersize=7, label=op_name)
    if key == 'prox_ratio_max':
        ax.axhline(1.0, color='k', ls='--', alpha=0.7, lw=1.2, label='cap (= 1)')
    ax.set_xlabel('Subdivision level')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(levels)
    ax.grid(True, which='both', alpha=0.25)
    last_handles, last_labels = ax.get_legend_handles_labels()

# Single shared legend below all three panels (4 entries fit in one row)
fig.legend(last_handles, last_labels,
           loc='lower center', bbox_to_anchor=(0.5, -0.04),
           ncol=4, frameon=True, framealpha=0.95,
           handlelength=2.0, columnspacing=1.6, fontsize=9.5)

fig.suptitle('Repeated-subdivision stability on ridge test set')
fig.savefig(OUT / 'repeated_subdivision.png', dpi=180, bbox_inches='tight')
plt.show()

RESULTS['repeated'] = dict(loop=agg_loop, constrained=agg_main, unconstrained=agg_unc)

RESULTS['repeated'] = dict(loop=agg_loop, constrained=agg_main, unconstrained=agg_unc)

## 14. Capacity sweepThe budget $C$ is swept over four values. Larger $C$ permits more aggressivefeature fitting, and smaller $C$ tightens proximity to Loop and widens thespectral safety margin. The right panel confirms that the measured proximityratio remains at or below the architectural cap at every tested budget, so theadmissible region below the cap is fully available.

In [ ]:
print("=" * 60)
print("C-SWEEP on ridges")
print("=" * 60)

C_VALUES = [0.1, 0.25, 0.5, 1.0]
sweep_results = {}
for C_val in C_VALUES:
    print(f"\n--- C = {C_val} ---")
    t0 = time.time()
    net_C, _ = cached_model(
        f'pns_sweep_C{C_val}',
        build_fn=lambda C_val=C_val: CorrectionNet(dim_in=5, hidden=32, n_layers=2, C=C_val),
        train_fn=lambda C_val=C_val: train_operator(
            RIDGE_TRAIN, RIDGE_TEST, C=C_val, weights=RIDGE_WEIGHTS,
            n_steps=steps(150), lr=2e-3, verbose=False, seed=2),
        meta=dict(C=C_val, n_steps=steps(150), seed=2))
    rows_C = evaluate_constrained_pair(RIDGE_TEST, net_C, C=C_val)
    near = paired_summary(loop_rows, rows_C, 'sdf_rmse_near')
    glob = paired_summary(loop_rows, rows_C, 'sdf_rmse')
    prox = max(r['prox_ratio_max'] for r in rows_C)
    sweep_results[C_val] = dict(
        sdf_near=near['mean_b'], sdf_global=glob['mean_b'],
        rel_near=near['rel_improvement_pct'], rel_global=glob['rel_improvement_pct'],
        prox_max=prox)
    print(f"  ({time.time()-t0:.1f}s)  near improvement = {near['rel_improvement_pct']:+.2f}%  "
          f"global = {glob['rel_improvement_pct']:+.2f}%  prox max = {prox:.4f}")
RESULTS['c_sweep'] = sweep_results

# Plot
# Plot
fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.4), constrained_layout=True)

# Left: relative improvement vs C
near_imp = [sweep_results[c]['rel_near'] for c in C_VALUES]
glob_imp = [sweep_results[c]['rel_global'] for c in C_VALUES]
axes[0].semilogx(C_VALUES, near_imp, 'o-', color=PALETTE[2], lw=1.8, markersize=8, label='near feature')
axes[0].semilogx(C_VALUES, glob_imp, 's-', color=PALETTE[1], lw=1.8, markersize=8, label='global')
axes[0].axhline(0, color='k', lw=0.6)
axes[0].set_xlabel(r'Perturbation budget $C$')
axes[0].set_ylabel('SDF RMSE improvement over Loop (%)')
axes[0].set_title('Approximation gain scales with $C$')
axes[0].grid(True, which='both', alpha=0.25)
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.16),
               ncol=2, framealpha=0.95, frameon=True,
               handlelength=2.0, columnspacing=1.6, fontsize=9)

# Right: structural margin saturates at the cap for every C
prox_max = [sweep_results[c]['prox_max'] for c in C_VALUES]
ax2 = axes[1]
ax2.semilogx(C_VALUES, prox_max, 'o-', color=PALETTE[2], lw=1.8, markersize=9,
             label='Max prox ratio (test set)', clip_on=False, zorder=4)
ax2.axhline(1.0, color='k', ls='--', alpha=0.7, lw=1.2,
            label='architectural cap', zorder=3)
ax2.fill_between(C_VALUES, [0]*len(C_VALUES), [1.0]*len(C_VALUES),
                  color=PALETTE[2], alpha=0.08, label='admissible region', zorder=1)
ax2.set_xlabel(r'Perturbation budget $C$')
ax2.set_ylabel(r'$\max_i \|q_i^\theta - q_i^0\| / (C h_i^2)$')
ax2.set_title('Structural margin holds at every $C$')
ax2.set_ylim(0, 1.15)
ax2.set_xlim(0.08, 1.3)
ax2.grid(True, which='both', alpha=0.25)
ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.16),
           ncol=3, framealpha=0.95, frameon=True,
           handlelength=2.0, columnspacing=1.6, fontsize=9)
for c, p in zip(C_VALUES, prox_max):
    ax2.annotate(f'{p:.3f}', xy=(c, p), xytext=(0, 8), textcoords='offset points',
                  ha='center', fontsize=8.5, color=PALETTE[2])

fig.savefig(OUT / 'C_sweep_ridges.png', dpi=180, bbox_inches='tight')
plt.show()



write_table('c_sweep', ['C', 'sdf_near', 'sdf_global', 'rel_near_pct',
                        'rel_global_pct', 'max_prox_ratio'],
            [dict(C=c, sdf_near=f"{sweep_results[c]['sdf_near']:.6f}",
                  sdf_global=f"{sweep_results[c]['sdf_global']:.6f}",
                  rel_near_pct=f"{sweep_results[c]['rel_near']:+.2f}",
                  rel_global_pct=f"{sweep_results[c]['rel_global']:+.2f}",
                  max_prox_ratio=f"{sweep_results[c]['prox_max']:.4f}")
             for c in C_VALUES])

### 14.1 Capacity controlA wider, longer-trained network tests whether the headline improvement islimited by network capacity or by the $C h^2$ envelope. If the confidenceinterval on the near-feature improvement overlaps the headline result, theoperator class is approximation-limited by the envelope rather than by capacity.

In [ ]:
print("=" * 70)
print("CAPACITY BOOST -- hidden=64, n_steps=500 (vs headline hidden=32, 250 steps)")
print("=" * 70)

net_big, hist_big = cached_model(
    'pns_capacity_h64',
    build_fn=lambda: CorrectionNet(dim_in=5, hidden=64, n_layers=2, C=0.5),
    train_fn=lambda: train_operator(
        RIDGE_TRAIN, RIDGE_TEST, C=0.5, weights=RIDGE_WEIGHTS,
        n_steps=steps(500), hidden=64, n_layers=2, lr=2e-3, seed=4, log_every=100),
    meta=dict(C=0.5, hidden=64, n_layers=2, n_steps=steps(500), seed=4))

big_rows = evaluate_constrained_pair(RIDGE_TEST, net_big, C=0.5)

# Side-by-side comparison
print()
print(f"{'Metric':<22s}  {'Headline (h=32, 250 steps)':>28s}  {'Boosted (h=64, 500 steps)':>28s}")
print('-' * 84)
for k, lbl in [('sdf_rmse_near', 'SDF near feature'), ('sdf_rmse', 'SDF global'),
               ('hausdorff_near','Hausdorff near'), ('normal_err_near','Normal err near')]:
    s_main = paired_summary(loop_rows, main_rows, k)
    s_big  = paired_summary(loop_rows, big_rows,  k)
    main_str = f"{s_main['rel_improvement_pct']:+5.2f}% [{s_main['ci_lo_pct']:+5.2f},{s_main['ci_hi_pct']:+5.2f}]"
    big_str  = f"{s_big['rel_improvement_pct']:+5.2f}% [{s_big['ci_lo_pct']:+5.2f},{s_big['ci_hi_pct']:+5.2f}]"
    print(f"  {lbl:<22s}  {main_str:>28s}  {big_str:>28s}")

prox_big = max(r['prox_ratio_max'] for r in big_rows)
print(f"\n  Boost prox ratio max: {prox_big:.4f}  (architectural cap = 1, must hold)")

print(f"\n  Parameter counts:  headline = {count_params(net_main):,}   "
      f"boost = {count_params(net_big):,}  ({count_params(net_big)/count_params(net_main):.1f}x)")

RESULTS['capacity_boost'] = {
    k: paired_summary(loop_rows, big_rows, k)
    for k in ['sdf_rmse', 'sdf_rmse_near', 'hausdorff_near', 'normal_err_near']
}
RESULTS['capacity_boost']['prox_max'] = prox_big
RESULTS['capacity_boost']['param_count'] = count_params(net_big)

## 15. Surface rendersShaded surface renders of the meshes each operator actually produces, at onesubdivision step and after four repeated steps, under shared camera and sharedcolour management. Annotation boxes report the final-level proximity ratio andthe maximum face-normal jump.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D    # noqa: F401  (registers 3d projection)
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from matplotlib import cm
import matplotlib.gridspec as gridspec

# Use the same money instance from §8
sample_for_render = RIDGE_TEST[mi]
m_render = sample_for_render['mesh']
sdf_for_render = sample_for_render['sdf']
pp_render = sample_for_render['params']


# ----- Ground-truth analytic surface, finely triangulated --------------------
def make_groundtruth_mesh(pp, half_size=1.0, nx=33, ny=33):
    cos_p, sin_p = math.cos(pp['phi']), math.sin(pp['phi'])
    x = torch.linspace(-half_size, half_size, nx, dtype=DTYPE)
    y = torch.linspace(-half_size, half_size, ny, dtype=DTYPE)
    X, Y = torch.meshgrid(x, y, indexing='ij')
    yp = -sin_p * (X - pp['cx']) + cos_p * (Y - pp['cy'])
    Z = pp['a'] * torch.exp(-pp['b'] * yp ** 2)
    V = torch.stack([X.flatten(), Y.flatten(), Z.flatten()], dim=1)
    F = []
    for i in range(nx - 1):
        for j in range(ny - 1):
            v00 = i*ny + j; v10 = (i+1)*ny + j
            v01 = i*ny + j+1; v11 = (i+1)*ny + j+1
            F.append([v00, v10, v11]); F.append([v00, v11, v01])
    return Mesh(V, torch.tensor(F, dtype=torch.long))

gt_mesh = make_groundtruth_mesh(pp_render, half_size=1.0, nx=33, ny=33)


# ----- Refinement chains (computed once) -------------------------------------
def make_chain(op_fn, m0, n_levels=4):
    chain = [m0]
    for _ in range(n_levels):
        chain.append(op_fn(chain[-1]))
    return chain

print("Generating refinement chains (4 levels x 3 operators)...")
t0 = time.time()
chain_loop = make_chain(subdivide_loop, m_render, 4)
chain_main = make_chain(lambda mm: subdivide_constrained(mm, net_main), m_render, 4)
chain_unc  = make_chain(lambda mm: subdivide_unconstrained(mm, net_unc),  m_render, 4)
print(f"  done in {time.time()-t0:.1f}s")


# ----- Compute structural stats for level-4 annotations ----------------------
def compute_chain_stats(parent_chain, C_arch=0.5):
    """Track max prox ratio across iterations and final-mesh max normal jump."""
    prox_ratios = []
    for L in range(len(parent_chain) - 1):
        parent = parent_chain[L]
        child  = parent_chain[L + 1]
        edge_keys, _, edge_opps = _build_full_edge_index(parent.F)
        q_0 = _new_edge_vertices_loop(parent.V, edge_keys, edge_opps)
        e_len = torch.tensor(
            [float(torch.norm(parent.V[a] - parent.V[b])) for a, b in edge_keys],
            dtype=DTYPE)
        h2 = (e_len ** 2).clamp_min(1e-12)
        n_old = parent.n_v
        q_t = child.V[n_old:]
        ratio_max = float(((q_t - q_0).norm(dim=1) / (C_arch * h2)).max())
        prox_ratios.append(ratio_max)
    final = parent_chain[-1]
    edges, edge_faces, _ = build_edge_data(final.F)
    Fn = _face_normals(final.V, final.F)
    cos_d = (Fn[edge_faces[:, 0]] * Fn[edge_faces[:, 1]]).sum(dim=1).clamp(-1, 1)
    max_jump = float(torch.acos(cos_d).max())
    return max(prox_ratios), max_jump

prox_loop_l4, jump_loop_l4 = compute_chain_stats(chain_loop)
prox_main_l4, jump_main_l4 = compute_chain_stats(chain_main)
prox_unc_l4,  jump_unc_l4  = compute_chain_stats(chain_unc)
print(f"  Loop:  max prox = {prox_loop_l4:.3f},  max normal jump = {jump_loop_l4:.3f} rad")
print(f"  S_t :  max prox = {prox_main_l4:.3f},  max normal jump = {jump_main_l4:.3f} rad")
print(f"  Unc :  max prox = {prox_unc_l4:.0f},     max normal jump = {jump_unc_l4:.3f} rad (pi = {math.pi:.3f})")


# ----- Rendering helpers -----------------------------------------------------
LIGHT_DIR = np.array([0.4, -0.4, 1.0])

def _shade_intensity(F, V):
    v0, v1, v2 = V[F[:, 0]], V[F[:, 1]], V[F[:, 2]]
    n = np.cross(v1 - v0, v2 - v0)
    n_unit = n / (np.linalg.norm(n, axis=1, keepdims=True) + 1e-12)
    light = LIGHT_DIR / np.linalg.norm(LIGHT_DIR)
    return np.clip(np.abs(n_unit @ light), 0.35, 1.0)

def render_gt_surface(ax, mesh):
    V = mesh.V.numpy(); F = mesh.F.numpy()
    intensity = _shade_intensity(F, V)
    base = np.array([0.78, 0.78, 0.82])
    fc = np.tile(np.array([*base, 1.0]), (F.shape[0], 1))
    fc[:, :3] *= (intensity[:, np.newaxis] * 0.7 + 0.3)
    poly = Poly3DCollection(V[F], facecolors=fc, edgecolors='none', linewidths=0.0, alpha=1.0)
    ax.add_collection3d(poly)

def render_op_surface(ax, mesh, sdf_fn, error_clip, edge_lw=0.0):
    V = mesh.V.numpy(); F = mesh.F.numpy()
    centroids = V[F].mean(axis=1)
    with torch.no_grad():
        sdf_vals = sdf_fn(torch.tensor(centroids, dtype=DTYPE)).abs().numpy()
    norm = plt.Normalize(vmin=0, vmax=error_clip)
    fc = cm.get_cmap('viridis')(norm(np.clip(sdf_vals, 0, error_clip)))
    fc[:, :3] *= (_shade_intensity(F, V)[:, np.newaxis] * 0.7 + 0.3)
    poly = Poly3DCollection(V[F], facecolors=fc,
                             edgecolors='black' if edge_lw > 0 else 'none',
                             linewidths=edge_lw, alpha=1.0)
    ax.add_collection3d(poly)
    return float(sdf_vals.max()), float(sdf_vals.mean())


def _configure_3d_axes(ax):
    ax.view_init(elev=22, azim=-58)
    ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05); ax.set_zlim(-0.05, 0.75)
    ax.set_box_aspect((2.1, 2.1, 0.65))
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.xaxis.pane.fill = False; ax.yaxis.pane.fill = False; ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('w'); ax.yaxis.pane.set_edgecolor('w'); ax.zaxis.pane.set_edgecolor('w')


# ----- Build the figure ------------------------------------------------------
ERROR_CLIP = 0.06
fig = plt.figure(figsize=(17.5, 8.5))
gs = gridspec.GridSpec(2, 4, figure=fig, width_ratios=[1.05, 1, 1, 1],
                       wspace=0.02, hspace=0.08, left=0.04, right=0.92,
                       top=0.93, bottom=0.04)

# Ground truth spans both rows
ax_gt = fig.add_subplot(gs[:, 0], projection='3d')
render_gt_surface(ax_gt, gt_mesh)
_configure_3d_axes(ax_gt)
ax_gt.set_box_aspect((2.1, 2.1, 1.4))   # taller since spans 2 rows
ax_gt.set_title(f"Ground truth\n$z = a\\,e^{{-b\\,y'^2}}$, $a={pp_render['a']:.2f}$, $b={pp_render['b']:.2f}$",
                 fontsize=11, pad=4)

# Operator panels
op_specs = [
    ('Loop', chain_loop),
    (r'Trained $S_\theta$', chain_main),
    ('Unconstrained neural foil', chain_unc),
]
levels_to_show = [1, 4]
op_axes = []
for i, lvl in enumerate(levels_to_show):
    for j, (name, chain) in enumerate(op_specs):
        ax = fig.add_subplot(gs[i, j+1], projection='3d')
        op_axes.append(ax)
        m_lvl = chain[lvl]
        edge_lw = 0.15 if lvl <= 1 else 0.0
        max_err, mean_err = render_op_surface(ax, m_lvl, sdf_for_render,
                                                error_clip=ERROR_CLIP, edge_lw=edge_lw)
        _configure_3d_axes(ax)
        title = f"{name}, level {lvl}\nmax|err| = {max_err:.4f}, mean = {mean_err:.4f}"
        ax.set_title(title, fontsize=10, pad=3)

# ----- Annotate level-4 panels with structural metrics -----------------------
def _annotate(ax, txt, color='black'):
    ax.text2D(0.04, 0.04, txt, transform=ax.transAxes, fontsize=9,
               verticalalignment='bottom', horizontalalignment='left',
               color=color,
               bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                          edgecolor='0.7', alpha=0.92))

# op_axes layout: [L1-Loop, L1-Sθ, L1-Unc, L4-Loop, L4-Sθ, L4-Unc]
_annotate(op_axes[3], f"normal jump = {jump_loop_l4:.2f} rad")
_annotate(op_axes[4], f"prox ratio $\\leq 1$;  normal jump = {jump_main_l4:.2f} rad",
          color='#2ca02c')
_annotate(op_axes[5],
          f"prox ratio $= {prox_unc_l4/1e4:.2f}\\times 10^4$\nnormal jump $\\approx \\pi$",
          color='#d62728')

# ----- Row labels on the left side ------------------------------------------
fig.text(0.005, 0.715, 'one subdivision step',
         fontsize=11, rotation=90, ha='center', va='center', fontweight='medium')
fig.text(0.005, 0.265, 'four repeated\nsubdivision steps',
         fontsize=11, rotation=90, ha='center', va='center', fontweight='medium')

# ----- Shared colorbar (height matched to panel rows, not full figure) ------
sm = cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(vmin=0, vmax=ERROR_CLIP))
sm.set_array([])
cax = fig.add_axes([0.935, 0.20, 0.011, 0.55])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label(r'SDF error $|d_\Sigma|$,  clipped at 0.06', fontsize=10)
cbar.ax.tick_params(labelsize=9)

plt.savefig(OUT / 'surface_renders.png', dpi=180, bbox_inches='tight')
plt.show()


# ----- Per-level summary printout --------------------------------------------
print()
print(f"Per-operator max|SDF error| at each level on this instance:")
print(f"  {'Level':<6s} {'Loop':>10s} {'S_theta':>10s} {'Unc neural foil':>18s}")
for L in range(5):
    e_l  = sdf_for_render(torch.tensor((chain_loop[L].V[chain_loop[L].F].mean(dim=1)).numpy(), dtype=DTYPE)).abs().max().item()
    e_m  = sdf_for_render(torch.tensor((chain_main[L].V[chain_main[L].F].mean(dim=1)).numpy(), dtype=DTYPE)).abs().max().item()
    e_u  = sdf_for_render(torch.tensor((chain_unc [L].V[chain_unc [L].F].mean(dim=1)).numpy(), dtype=DTYPE)).abs().max().item()
    print(f"  {L:<6d} {e_l:10.4f} {e_m:10.4f} {e_u:18.4f}")

## 16. Saddle controlThe same experiment on smooth saddle surfaces $z = a(x^2 - y^2)$. Saddles arealready well approximated by Loop, with truncation error that is small and notlocalised, so there is little for a curvature-gated correction to do and thegate stays close to zero. A null result here is the expected behaviour and isreported as such. The gains on ridges are feature-driven rather than universal.

In [ ]:
print("=" * 60)
print("SADDLE TRAINING (secondary)")
print("=" * 60)
net_saddle, _ = cached_model(
    'pns_saddle',
    build_fn=lambda: CorrectionNet(dim_in=5, hidden=32, n_layers=2, C=0.5),
    train_fn=lambda: train_operator(
        SADDLE_TRAIN, SADDLE_TEST, C=0.5, weights=RIDGE_WEIGHTS,
        n_steps=steps(150), lr=2e-3, seed=3, verbose=False),
    meta=dict(C=0.5, n_steps=steps(150), seed=3))

loop_rows_sad = evaluate_loop_pair(SADDLE_TEST)
main_rows_sad = evaluate_constrained_pair(SADDLE_TEST, net_saddle, C=0.5)

print()
print(f"{'Metric':<22s}  {'Loop':>10s}  {'trained S_theta':>16s}  {'improvement':>12s}")
print('-' * 64)
saddle_summary = {}
for k in ['sdf_rmse_near', 'sdf_rmse', 'normal_err_near']:
    s = paired_summary(loop_rows_sad, main_rows_sad, k)
    saddle_summary[k] = s
    print(f"  {k:<22s}  {s['mean_a']:10.5f}  {s['mean_b']:16.5f}  "
          f"{s['rel_improvement_pct']:+11.2f}% [{s['ci_lo_pct']:+.2f}%, {s['ci_hi_pct']:+.2f}%]")
prox_sad = max(r['prox_ratio_max'] for r in main_rows_sad)
print(f"  prox ratio max         : trained = {prox_sad:.4f}  (must be <= 1)")

RESULTS['saddle'] = saddle_summary

## 17. Cross-family generalisationThe flat-ridge family is the only one the operator was trained on. Evaluatingthe same trained network, without retraining, on a curved-ridge-on-sphere familytests whether the gains come from learned ridge fitting or from architecturalfeature localisation. The relevant question is whether the gate fires on theridge while staying quiet on the sphere's background curvature.

In [ ]:
# ---- Sphere-ridge dataset (no training, just evaluation of the existing net_main) ----
def make_sphere_ridge_sdf(a=0.10, b=20.0, phi=0.0, R=1.0):
    cos_p, sin_p = math.cos(phi), math.sin(phi)
    def sdf(p):
        r = torch.norm(p, dim=-1).clamp_min(1e-9)
        ux = p[..., 0] / r; uy = p[..., 1] / r
        y_unit = -sin_p * ux + cos_p * uy
        y_clamped = y_unit.clamp(-1+1e-7, 1-1e-7)
        alpha = torch.asin(y_clamped)
        ridge = a * torch.exp(-b * alpha ** 2)
        f = r - R - ridge
        # analytic gradient norm
        dridge_dalpha = -2 * a * b * alpha * torch.exp(-b * alpha ** 2)
        denom = torch.sqrt((1 - y_clamped ** 2).clamp_min(1e-7))
        e_yhat = torch.zeros_like(p)
        e_yhat[..., 0] = -sin_p; e_yhat[..., 1] = cos_p
        dyu_dp = (e_yhat - y_unit.unsqueeze(-1) * p / r.unsqueeze(-1)) / r.unsqueeze(-1)
        dalpha_dp = dyu_dp / denom.unsqueeze(-1)
        df_dp = (p / r.unsqueeze(-1)) - dridge_dalpha.unsqueeze(-1) * dalpha_dp
        gnorm = torch.norm(df_dp, dim=-1).clamp_min(1e-9)
        return f / gnorm
    return sdf


def lift_to_sphere_ridge(Vxy, a, b, phi, R=1.0, half_angle=0.5):
    cos_p, sin_p = math.cos(phi), math.sin(phi)
    u = Vxy[:, 0]; v = Vxy[:, 1]
    theta = u * half_angle; phi_lat = v * half_angle
    cx = torch.cos(phi_lat) * torch.cos(theta)
    cy = torch.cos(phi_lat) * torch.sin(theta)
    cz = torch.sin(phi_lat)
    y_unit = -sin_p * cx + cos_p * cy
    alpha = torch.asin(y_unit.clamp(-1+1e-7, 1-1e-7))
    bump = a * torch.exp(-b * alpha ** 2)
    r = R + bump
    return torch.stack([r * cx, r * cy, r * cz], dim=1)


def make_sphere_ridge_dataset(n_instances, seed_base=2000, R=1.0, half_angle=0.5):
    samples = []
    rng = np.random.default_rng(seed_base)
    for i in range(n_instances):
        a   = float(rng.uniform(0.04, 0.12))
        b   = float(rng.uniform(15.0, 60.0))
        phi = float(rng.uniform(0.0, math.pi))
        Vxy, F = grid_patch(nx=9, ny=9, half_size=1.0, jitter=0.03, seed=seed_base + i)
        V = lift_to_sphere_ridge(Vxy, a=a, b=b, phi=phi, R=R, half_angle=half_angle)
        m = Mesh(V, F)
        sdf_fn = make_sphere_ridge_sdf(a=a, b=b, phi=phi, R=R)
        samples.append(dict(mesh=m, sdf=sdf_fn,
                            params=dict(a=a, b=b, phi=phi, R=R, half_angle=half_angle,
                                         family='sphere'),
                            name=f'sphridge_{i}'))
    return samples


SPHERE_TEST = make_sphere_ridge_dataset(10, seed_base=2999)
print(f"Sphere-ridge test set: {len(SPHERE_TEST)} instances\n")


# ---- Cross-family evaluation: Loop vs. trained S_theta on sphere ridges ----
@torch.no_grad()
def evaluate_sphere(samples, q_fn):
    out = []
    for s in samples:
        m = s['mesh']
        edges, edge_faces, _ = build_edge_data(m.F)
        q = q_fn(m)
        d = s['sdf'](q)
        # near-feature mask via spherical alpha
        cos_p, sin_p = math.cos(s['params']['phi']), math.sin(s['params']['phi'])
        r = q.norm(dim=1).clamp_min(1e-9)
        y_unit = (-sin_p * q[:, 0] + cos_p * q[:, 1]) / r
        alpha = torch.asin(y_unit.clamp(-1+1e-7, 1-1e-7))
        nm = (alpha.abs() < 0.10)
        out.append(dict(
            sdf_rmse  = float(torch.sqrt((d**2).mean())),
            sdf_rmse_near = float(torch.sqrt((d[nm]**2).mean())) if nm.any() else float('nan'),
            hausdorff = float(d.abs().max()),
        ))
    return out


loop_rows_sph = evaluate_sphere(SPHERE_TEST,
    lambda m: loop_edge_vertices(m.V, *build_edge_data(m.F)[::2]))
pns_rows_sph  = evaluate_sphere(SPHERE_TEST,
    lambda m: constrained_edge_vertices(m, net_main)[0])

def _mean(rs, k): return float(np.mean([r[k] for r in rs]))
def _rel(a, b):  return (a - b) / a * 100 if a > 0 else 0.0

print(f"{'metric':<20s} {'Loop':>12s} {'Trained S_theta':>18s} {'rel %':>10s}")
print('-' * 64)
sphere_rows = []
for k in ('sdf_rmse', 'sdf_rmse_near', 'hausdorff'):
    a = _mean(loop_rows_sph, k); b = _mean(pns_rows_sph, k)
    print(f"{k:<20s} {a:12.5f} {b:18.5f} {_rel(a, b):+9.1f}%")
    sphere_rows.append(dict(metric=k, loop=f'{a:.6f}', pns=f'{b:.6f}',
                            improvement_pct=f'{_rel(a, b):+.2f}'))
write_table('cross_family_sphere_ridge',
            ['metric', 'loop', 'pns', 'improvement_pct'], sphere_rows)
RESULTS['cross_family'] = sphere_rows

print()
print("Interpretation:")
print("  - The constrained operator was trained ONLY on flat ridges; this")
print("    is a held-out family it has never seen.")
print("  - Performance on near-feature SDF probes whether the curvature gate")
print("    fires on the ridge but stays quiet on the sphere's background curvature.")
print("  - The operator class is unchanged on this family, so the architectural")
print("    guarantees certified in Section 6 continue to hold without retraining.")

## 18. SummaryEvery number produced above is collected here and written to `Results/metrics`,so the reported values can be checked without re-running anything.

In [ ]:
print('=' * 78)
print('PROXIMITY-PRESERVING NEURAL SUBDIVISION  --  RUN SUMMARY')
print('=' * 78)
hp = RESULTS['headline']['paired']

print()
print(f"  Headline ridge experiment (10 held-out instances, 95% CI on paired improvement):")
for k, lbl in [('sdf_rmse_near', 'SDF RMSE near feature'), ('sdf_rmse', 'SDF RMSE global'),
               ('hausdorff_near','Hausdorff near'), ('normal_err_near','Normal err near')]:
    s = hp[k]
    print(f"    {lbl:<24s}  Loop = {s['mean_a']:.5f}  S_t = {s['mean_b']:.5f}  "
          f"improvement = {s['rel_improvement_pct']:+.2f}% [{s['ci_lo_pct']:+.2f}%, {s['ci_hi_pct']:+.2f}%]")
print(f"    Max prox ratio over test set : {max(r['prox_ratio_max'] for r in main_rows):.4f}  (must be <= 1)")
print()
print(f"  C-sweep (near-feature improvement / global / max prox ratio):")
for c in [0.1, 0.25, 0.5, 1.0]:
    sw = RESULTS['c_sweep'][c]
    print(f"    C = {c:4.2f}:  near {sw['rel_near']:+5.2f}%   global {sw['rel_global']:+5.2f}%   "
          f"prox max = {sw['prox_max']:.4f}")
print()
print(f"  Repeated-subdivision stability (level 4):")
print(f"    Loop          : SDF = {RESULTS['repeated']['loop'][4]['sdf_rmse']:.5f}")
print(f"    Constrained   : SDF = {RESULTS['repeated']['constrained'][4]['sdf_rmse']:.5f}, "
      f"max prox = {RESULTS['repeated']['constrained'][4]['prox_ratio_max']:.4f}")
print(f"    Unconstrained : SDF = {RESULTS['repeated']['unconstrained'][4]['sdf_rmse']:.5f}, "
      f"max prox = {RESULTS['repeated']['unconstrained'][4]['prox_ratio_max']:.4f}")
print()
print(f"  Saddle secondary (4 held-out):")
ss = RESULTS['saddle']
print(f"    SDF near improvement = {ss['sdf_rmse_near']['rel_improvement_pct']:+.2f}% "
      f"[{ss['sdf_rmse_near']['ci_lo_pct']:+.2f}%, {ss['sdf_rmse_near']['ci_hi_pct']:+.2f}%]")

cb = RESULTS['capacity_boost']
print()
print(f"  Capacity boost (hidden=64, 500 steps):")
print(f"    SDF near improvement = {cb['sdf_rmse_near']['rel_improvement_pct']:+.2f}% "
      f"[{cb['sdf_rmse_near']['ci_lo_pct']:+.2f}%, {cb['sdf_rmse_near']['ci_hi_pct']:+.2f}%]  "
      f"(headline = {hp['sdf_rmse_near']['rel_improvement_pct']:+.2f}%)")
print(f"    SDF global improvement = {cb['sdf_rmse']['rel_improvement_pct']:+.2f}%   "
      f"prox max = {cb['prox_max']:.4f}  ({cb['param_count']:,} params)")
print('=' * 78)


# ---- Persist the complete run record ----
(METRIC_DIR / 'metrics_summary.json').write_text(
    json.dumps(to_safe(dict(RESULTS)), indent=2))
print()
print("  Results/metrics/metrics_summary.json   complete run record")
for name in sorted(p.name for p in METRIC_DIR.glob('*.csv')):
    print(f"  Results/metrics/{name}")
print()
for name in sorted(p.name for p in FIG_DIR.glob('*.png')):
    print(f"  Results/figures/{name}")
print()
for name in sorted(p.name for p in MODEL_DIR.glob('*.pt')):
    print(f"  Results/models/{name}")
print('=' * 78)